In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S03_ols"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 3 — Regresión Lineal Simple (OLS)

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios
**Laboratorio de replicación:** Galton (1886), *Regression Towards Mediocrity in Hereditary Stature* — el estudio que dio nombre a la «regresión» — y caso de negocio *Advertising* (ISLR).

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


---

## 1. Objetivos de aprendizaje

Al terminar la sesión, el estudiante:

- Ajusta un modelo **OLS de un predictor** e **interpreta** los coeficientes β₀ y β₁ en unidades de negocio.
- Valida los **supuestos de Gauss-Markov** mediante el análisis de residuales.
- Distingue la **inferencia sobre β** (error estándar, t-test, IC) de la **predicción de ŷ** (IC de la media vs. intervalo de predicción).
- Evalúa el modelo **fuera de muestra** con partición train/test y las métricas RMSE / MAE.
- Comunica el resultado a una audiencia **no técnica**.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —3.1 a 3.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 2)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **3.1** | ¿Por qué no basta la correlación estudiada en S02? | Sin celdas: se trabaja en clase |
| **3.2** | ¿Qué es una regresión lineal simple y qué decisión habilita? | «Teoría guiada» (celdas 12 a 20) |
| **3.3** | ¿Cómo se obtiene la recta óptima? | «Teoría guiada» (celdas 12 a 20) |
| **3.4** | ¿Qué calidad de ajuste ofrece y qué grado de confianza merece? | «Teoría guiada» (celdas 12 a 20) |
| **3.5** | ¿Se sostiene con datos reales? La réplica de Galton (1886) — laboratorio pasos 0–5 | «Réplica de Galton» (celdas 21 a 46) |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **3.6** | ¿Cuándo se puede confiar en esta recta? | «Supuestos» (celdas 74 a 94) |
| **3.7** | ¿Cómo se verifica que el desempeño es real? | «Estabilidad fuera de muestra» (celda 43) y métricas del caso de negocio |
| **3.8** | ¿Qué decisión habilita? El caso Advertising — laboratorio paso 6 | «Laboratorio de negocio» (celdas 47 a 58) |
| **3.9** | ¿Qué no se puede afirmar, y qué sigue en S04? | «Cierre» (celda 96) |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **se ejecuta**: explica cada paso. Convenciones:

- **❓ Qué se quiere averiguar** abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** precede a cada celda de código; **📖 Cómo se lee esta salida** sigue a cada resultado numérico clave. **💡** añade intuición y **⚠️** marca un supuesto o alerta.
- El alumno **realiza la mecánica de forma manual** y la verifica contra la librería con `assert`: **🖐️ Cálculo manual**, **🧮 Matemática en el cuerpo** (derivación en LaTeX), **✅ Verificación desde la base** (recomputa el resultado y lo cruza con el Excel) y **🧱 Construcción del OLS desde cero**.
- **📄 En el paper** indica la procedencia exacta de cada resultado replicado (fuente, sección, página).
- La **«Sección 8» (Supuestos)** ejecuta los diagnósticos; su teoría vive en la guía de supuestos de la sesión.
- **Valor operativo vs. benchmark:** las cifras que se ejecutan aquí son las del **venv** (prevalecen); las del **paper** (p. ej. el 2/3 de Galton) se citan como *benchmark* etiquetado. Pequeñas diferencias provienen del método (tabulación de 1886 vs. OLS de fila) y caen dentro de tolerancia.
- **Convención Excel:** los resultados del modelo se vuelcan a `resultados/S03_resultados.xlsx` y las **figuras de resultados se generan leyendo ese Excel**. Los bloques de descomposición y de supuestos **no escriben** en él.

Materiales hermanos: `laboratorio/GUIA_LABORATORIO_S03.docx`, `plantillas/reporte_regresion_gerencial.docx`, `plantillas/guia_residuales.docx`, `evaluacion/drills.docx`, `evaluacion/entregable.docx`, el cuaderno de la sesión y la fuente canónica de supuestos la guía de supuestos de la sesión.

## Preparación del entorno

La celda siguiente instala las librerías con versiones fijadas **solo cuando el notebook se ejecuta en Google Colab**. En una ejecución local con el entorno del curso ya configurado, se omite automáticamente.

**🔎 Qué hace este código.** Instala, **solo en Google Colab**, las librerías de la sesión **paquete a paquete** con las versiones **certificadas** en la matriz de versiones certificada del curso (autoridad del curso). Si un tag no resuelve, **solo ese paquete** se reintenta sin fijar versión: los demás conservan su pin (un `%pip install` único sería todo-o-nada y dejaría al alumno sin ninguna librería). Al final imprime la tabla **certificada → instalada**, que también se imprime en ejecución local para documentar el entorno real.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "seaborn": "0.13.2",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Importa las librerías, define la paleta UPC y el ayudante `mostrar()` (guarda cada figura como PNG y la muestra en el cuaderno). El backend `Agg` genera figuras sin ventana gráfica.

In [ ]:
# Configuración, imports y ayudantes
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # backend headless: las figuras se guardan como PNG
import matplotlib.pyplot as plt
from IPython.display import Image, display
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

EN_COLAB = "google.colab" in sys.modules

# Paleta del curso
UPC_ROJO, UPC_TINTA, UPC_GRIS = "#C8102E", "#1F2A44", "#8A8D8F"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def mostrar(fig, ruta):
    "Guarda la figura como PNG (140 dpi) y la muestra en el cuaderno."
    fig.tight_layout()
    fig.savefig(ruta, dpi=140, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(ruta)))

print("Librerias cargadas. statsmodels y scikit-learn listos para OLS.")

**🔎 Qué hace este código.** Localiza la carpeta `data/` en local o el directorio de trabajo en Colab, y fija las rutas de `resultados/` (Excel) y `figuras/`. Deja lista la variable `XLSX`, que apunta al Excel de contrato.

In [ ]:
# Localizacion de rutas de la sesion (funciona en local y en Colab)
def _detectar_base():
    cwd = Path.cwd()
    for cand in (cwd.parent, cwd, cwd / "Sesiones" / "S03_ols"):
        if (cand / "data" / "descargar_datos.py").exists():
            return cand
    return cwd.parent

if EN_COLAB:
    BASE = Path.cwd()
    DATA_DIR = BASE
else:
    BASE = _detectar_base()
    DATA_DIR = BASE / "data"

RESULTS_DIR = BASE / "resultados"; RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = BASE / "figuras"; FIG_DIR.mkdir(parents=True, exist_ok=True)
XLSX = RESULTS_DIR / "S03_resultados.xlsx"

print("Entorno Colab:", EN_COLAB)
print("Datos      ->", DATA_DIR)
print("Resultados ->", RESULTS_DIR)
print("Figuras    ->", FIG_DIR)

**🔎 Qué hace este código.** Carga los dos datasets **verificados**: `GaltonFamilies` (réplica del paper) y `Advertising` (caso de negocio). La fuente preferente es el script del curso `data/descargar_datos.py` (idempotente, con checksum SHA256); en Colab, si el script no está, recurre a las mismas fuentes verificadas.

In [ ]:
# Carga de datos verificados (Galton y Advertising)
GALTON_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/HistData/GaltonFamilies.csv"
ADV_URLS = [
    "https://www.statlearning.com/s/Advertising.csv",
    "https://raw.githubusercontent.com/nguyen-toan/ISLR/master/dataset/Advertising.csv",
]

def cargar_datos():
    try:
        if str(DATA_DIR) not in sys.path:
            sys.path.insert(0, str(DATA_DIR))
        import descargar_datos as dd
        return dd.obtener_galton(), dd.obtener_advertising()
    except Exception as e:  # noqa: BLE001 (fallback para el Colab del alumno)
        print("Aviso: descargar_datos.py no disponible; se usa descarga directa.")
        print("  Detalle:", repr(e))
        galton = pd.read_csv(GALTON_URL)
        adv = None
        for u in ADV_URLS:
            try:
                adv = pd.read_csv(u); break
            except Exception:
                continue
        return galton, adv

galton, adv = cargar_datos()
print("\nGaltonFamilies:", galton.shape, "->", list(galton.columns))
print("Advertising   :", adv.shape, "->", list(adv.columns))

## 3.2 a 3.4 — Teoría guiada: qué afirma el modelo, cómo se obtiene y cuánta confianza merece

Cada bloque combina una idea, una **minidemostración** ejecutable y su **lectura de negocio**.

### 3.1 De la correlación al modelo lineal

En la Sesión 2 la **correlación** `r` resumía la *fuerza* y la *dirección* de la asociación entre dos variables, pero no cuantificaba **cuánto** cambia una respuesta cuando el predictor sube una unidad. La **regresión lineal simple** da ese salto: describe la media esperada de una respuesta cuantitativa `Y` en función de **un** predictor `X` mediante una recta

$$Y = \beta_0 + \beta_1 X + \varepsilon,$$

donde `ε` es el error aleatorio. La **recta ajustada** `ŷ = b₀ + b₁X` es la mejor estimación de la media de `Y` para cada `X`.

**Lectura de negocio:** convierte una asociación («a más inversión publicitaria, más ventas») en una regla accionable y medible («cada 1000 USD de inversión añaden X unidades de venta»), base de presupuestos, pronósticos y decisiones de precio.

### 3.2 Mínimos cuadrados ordinarios (OLS): intuición geométrica

OLS elige la recta que **minimiza la suma de los residuales al cuadrado** `SSE = Σ(yᵢ − ŷᵢ)²`; geométricamente, la que deja la menor distancia vertical cuadrada total a los puntos. Su solución cerrada conecta directamente con la correlación de S02: `b₁ = r·(sᵧ/sₓ)` y `b₀ = ȳ − b₁x̄`. Por construcción la recta pasa por el centroide `(x̄, ȳ)`. Bajo los supuestos de Gauss-Markov, OLS es **insesgado y de mínima varianza** entre los estimadores lineales (teorema BLUE): rápido, interpretable y auditable.

### 🧮 Matemática en el cuerpo — ecuaciones normales, solución OLS y R²

OLS minimiza la suma de residuales al cuadrado. Derivando `SSE(β₀,β₁) = Σ(yᵢ − β₀ − β₁xᵢ)²` respecto de β₀ y β₁ e igualando a cero se obtienen las **ecuaciones normales**:

$$\sum_i (y_i - \beta_0 - \beta_1 x_i) = 0, \qquad \sum_i x_i\,(y_i - \beta_0 - \beta_1 x_i) = 0.$$

Resolviéndolas resulta la **solución OLS** en forma escalar:

$$\hat{\beta}_1 = \frac{\sum_i (x_i-\bar{x})(y_i-\bar{y})}{\sum_i (x_i-\bar{x})^2} = r\,\frac{s_y}{s_x}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\,\bar{x}.$$

En forma **matricial**, con `X = [1, x]`, la misma solución es `β = (XᵀX)⁻¹Xᵀy` (la que reconstruye la «Sección 7»). Esa inversa es la **equivalencia matemática**, no la implementación: por estabilidad numérica `statsmodels` resuelve el sistema con la pseudoinversa (`pinv`, vía descomposición SVD) y `scikit-learn` con `lstsq`, sin formar nunca `(XᵀX)⁻¹` —la ruta de las ecuaciones normales es la menos estable numéricamente—. La bondad de ajuste se resume con el **coeficiente de determinación**:

$$R^2 = 1 - \frac{\text{SSE}}{\text{SST}}, \qquad \text{SSE}=\sum_i(y_i-\hat{y}_i)^2,\quad \text{SST}=\sum_i(y_i-\bar{y})^2.$$

En regresión **simple** `R² = r²`. El bloque 🖐️ siguiente reconstruye estas tres fórmulas y las verifica contra `statsmodels`.

**🔎 Qué hace este código.** Comprueba, sobre datos sintéticos pequeños, que la pendiente OLS coincide con `r·(sᵧ/sₓ)`: la misma relación que une la correlación de S02 con la regresión.

In [ ]:
# Mini-demo: la pendiente OLS es r*(sy/sx). Datos sinteticos pequenos.
rng = np.random.default_rng(7)
x_demo = rng.normal(50, 10, 60)
y_demo = 8 + 0.5 * x_demo + rng.normal(0, 4, 60)

r_demo = np.corrcoef(x_demo, y_demo)[0, 1]
b1_formula = r_demo * (y_demo.std(ddof=1) / x_demo.std(ddof=1))
b1_ols = sm.OLS(y_demo, sm.add_constant(x_demo)).fit().params[1]
print(f"r                 = {r_demo:.4f}")
print(f"b1 = r*(sy/sx)    = {b1_formula:.4f}")
print(f"b1 por OLS        = {b1_ols:.4f}   (coinciden)")

**📖 Cómo se lee.** La pendiente calculada como `r·(sᵧ/sₓ)` y la que devuelve OLS coinciden dígito a dígito: la regresión **reescala** la correlación por el cociente de desviaciones. Es el puente exacto entre S02 y S03.

### 3.3 Supuestos de Gauss-Markov (resumen)

OLS es BLUE bajo: **linealidad** en los parámetros, **exogeneidad** `E[ε|X]=0`, **homocedasticidad** (varianza constante del error), **no autocorrelación** y **variación en X**. La **normalidad** de los residuales no es requisito de Gauss-Markov, pero se asume para que las pruebas t/F e intervalos sean exactos en muestras pequeñas. El diagnóstico completo —con pruebas y gráficos— vive en la «Sección 8»; su fuente canónica es la guía de supuestos de la sesión.

### 3.4 Cuarteto de Anscombe: por qué hay que graficar

Frank Anscombe (1973) construyó cuatro conjuntos con estadísticos resumen **casi idénticos** —misma media de `X` e `Y`, misma varianza, misma correlación (r ≈ 0,816) y prácticamente la misma recta— pero cuyos diagramas de dispersión son radicalmente distintos. Es el argumento visual de por qué **los números resumen no bastan**. Aquí se computan sus estadísticos; en la «Sección 8» se **grafican** las cuatro nubes.

**❓ Qué se quiere averiguar.** ¿Basta con los estadísticos de resumen —media, pendiente, correlación, R²— para saber si un modelo describe bien unos datos?

- **Por qué importa:** en el trabajo real se reciben tablas de indicadores, no nubes de puntos. Si los resúmenes bastaran, bastaría con leerlos.
- **Antes de mirar el resultado:** si los cuatro conjuntos arrojan **cifras distintas**, los resúmenes discriminan y sirven por sí solos. Si resultan **idénticos**, entonces cuatro realidades distintas producen el mismo informe, y ningún indicador puede sustituir al gráfico.

**🔎 Qué hace este código.** Ajusta un OLS a cada uno de los cuatro datasets de Anscombe y construye la tabla de estadísticos resumen: media de `x`, media de `y`, pendiente, r y —para que ninguna cifra publicada quede sin celda— **intercepto β₀ y R²**. La tabla alimenta la hoja `anscombe` del Excel.

In [ ]:
# Cuarteto de Anscombe: estadisticos resumen por dataset (identicos pese a nubes distintas)
anscombe = sns.load_dataset("anscombe")

filas = []
for nombre, g in anscombe.groupby("dataset"):
    x, y = g["x"].to_numpy(), g["y"].to_numpy()
    res_ans = sm.OLS(y, sm.add_constant(x)).fit()
    filas.append({"dataset": nombre, "media_x": x.mean(), "media_y": y.mean(),
                  "pendiente": float(res_ans.params[1]), "r": np.corrcoef(x, y)[0, 1],
                  "intercepto_b0": float(res_ans.params[0]), "r2": float(res_ans.rsquared)})
anscombe_stats = pd.DataFrame(filas)
print(anscombe_stats.round(4).to_string(index=False))
print("\nMisma media, misma pendiente (~0.50), misma r (~0.816), mismo b0 (~3.00) y mismo R2 (~0.67)...")
print("pero se ven distintos.")

**📖 Cómo se lee.** Los cuatro comparten media de `x` (9), media de `y` (7,5), pendiente (~0,50), r (~0,816), intercepto (~3,00) y R² (~0,67): el resumen numérico es indistinguible. La lección la dará la figura de la «Sección 8».

### 3.5 Inferencia: precisión de β, R² e intervalos

Una vez ajustada la recta, se cuantifica cuánto confiar en ella: el **error estándar de β₁** mide su precisión; el **t-test** contrasta `H₀: β₁ = 0`; el **R²** es la fracción de varianza explicada; y ante un `x₀` conviven el **IC de la media** (estrecho) y el **intervalo de predicción** (mucho más ancho). El bloque 🧮 siguiente formaliza esas cantidades.

### 🧮 Matemática en el cuerpo — error estándar de β₁, estadístico t e intervalo de confianza (subsección 3.6)

Con la varianza del error estimada por `s² = SSE/(n−2)` y `Sₓₓ = Σ(xᵢ−x̄)²`, el **error estándar de la pendiente** es

$$\operatorname{SE}(\hat{\beta}_1) = \frac{s}{\sqrt{S_{xx}}}, \qquad s = \sqrt{\tfrac{\text{SSE}}{n-2}}.$$

El **estadístico t** contrasta `H₀: β₁ = 0` y su cuadrado es el estadístico `F` de la regresión simple:

$$t = \frac{\hat{\beta}_1}{\operatorname{SE}(\hat{\beta}_1)}, \qquad F = t^2.$$

El **intervalo de confianza al 95 %** de β₁ usa el cuantil de la `t` de Student con `n−2` grados de libertad:

$$\hat{\beta}_1 \pm t_{0{,}975,\,n-2}\;\operatorname{SE}(\hat{\beta}_1).$$

Estas fórmulas se recomputan y verifican contra el Excel en los bloques ✅ (Sección 6.1) y 🧱 (Sección 7).

### 3.6 Train/test y RMSE/MAE: por qué el R² in-sample induce a error

El R² se calcula **sobre los mismos datos** que ajustaron la recta, así que **sobrestima** el desempeño futuro (**sobreajuste**). La defensa es medir el error **fuera de muestra**: se separa un conjunto de **entrenamiento** (ajusta) y uno de **prueba** (evalúa sobre datos no vistos) y se reportan el **RMSE** (penaliza más los errores grandes) y el **MAE** (trata todos los errores por igual), ambos **en las unidades de `Y`**. Un modelo se valora por su desempeño con datos futuros, no por lo que «explica» hoy. Este esquema se introduce en S03 y se reutiliza en S04-S14.

## 3.5 — ¿Se sostiene con datos reales? La réplica de Galton (1886)

**Paper:** Galton, F. (1886). *Regression Towards Mediocrity in Hereditary Stature*. Journal of the Anthropological Institute of Great Britain and Ireland, 15, 246-263.

Galton midió la estatura de **205 familias** y observó que los hijos de padres extremadamente altos tienden a ser altos, pero **menos** altos que sus padres (y viceversa): las estaturas de la descendencia «regresan hacia la mediocridad» (la media). De ahí nació el término **regresión**. Galton reportó que la altura del hijo cambia en promedio **dos tercios (≈ 0,667)** de la desviación de la mid-parent respecto a la media: el ancestro directo de la pendiente β₁ del OLS moderno.

### 4.1 El dataset y la advertencia de género

`midparentHeight` es la altura combinada de los padres, y **ya incorpora** un ajuste por sexo del lado materno:

$$\text{midparentHeight} = \frac{\text{father} + 1{,}08 \cdot \text{mother}}{2}.$$

Galton multiplicó la altura de cada mujer por **1,08** para expresarla en «equivalente masculino» (lo que llamó *transmutación*). En cambio, `childHeight` está **cruda** (mezcla hijos e hijas en su escala natural). Esa asimetría se aprovecha como momento de enseñanza más abajo.

### 4.0 Qué preguntaba Galton, y por qué usó lo que usó

**💡 Antes de trabajar con los datos.** Una réplica sin esta pregunta se convierte en mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba el autor** y **por qué eligió cada pieza de su método**, que es de donde procede el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El objetivo no era inventar la regresión.** En la primera página del memoir Galton declara: «My object is to place beyond doubt the existence of a simple and far-reaching law that governs the hereditary transmission of […] every one of those simple qualities which all possess» (p. 246). La pregunta era **biológica**: ¿existe una ley simple y cuantificable que diga **cuánto** se transmite un rasgo de una generación a la siguiente? La palabra «regresión» fue el nombre que dio al **hallazgo**, no el objetivo del trabajo.

Y la respuesta tenía consecuencias: si la transmisión fuera **íntegra**, los descendientes de progenitores extremos serían igual de extremos y la dispersión de estaturas crecería generación tras generación; si es **parcial**, cada peculiaridad familiar se reabsorbe y la población permanece estable. Por eso Galton necesitaba un **número**, no una impresión.

**Por qué la estatura** (p. 249). Se mide con facilidad, permanece constante durante treinta y cinco años de vida adulta, depende poco de la crianza y apenas influye sobre la mortalidad. Y, sobre todo, no es un rasgo simple sino la **suma de más de cien partes del cuerpo**: por eso su distribución poblacional resulta casi perfectamente regular. Es el argumento que hoy se enuncia como teorema central del límite — eligió la variable **cuya distribución sabía bien comportada**.

**Por qué la mid-parent** (pp. 247 y 250). Primero llevó ambas estaturas a una escala común (la *transmutación* ×1,08, que se verifica en la celda siguiente). Después **comprobó que podía descartar el resto**: ordenó las familias por la diferencia de altura entre los progenitores y vio que esa diferencia no alteraba la distribución de los hijos, de donde «the average height of the two parents […] is all we need care to know about them». **Verificó el supuesto antes de apoyarse en él** — exactamente lo que se hará más abajo con los residuales.

**Por qué una tabla y una recta trazada gráficamente, y no una fórmula** (p. 248). Porque la fórmula no existía: la correlación se formaliza diez años después (Pearson, 1896) y los mínimos cuadrados se usaban entonces para ajustar órbitas, no para relacionar dos rasgos de seres vivos. Galton tabuló las estaturas en una tabla de doble entrada, resumió cada fila con **la mediana** —robusta y legible directamente de los conteos, decisivo cuando todo se calcula manualmente— y trazó la recta sobre el papel: su pendiente es el 2/3.

**⚠️ De ahí la diferencia que aparecerá al comparar.** Este cuaderno sustituye ese ajuste gráfico por un **OLS sobre las 934 filas individuales**: por eso obtiene 0,637 y no 0,667. La diferencia proviene del **método de ajuste**, no de un error, y lo que sí se reproduce intacto es la conclusión que responde la pregunta de Galton: **la pendiente es menor que 1**.


**🔎 Qué hace este código.** Muestra la estructura del dataset y **verifica la fórmula de la mid-parent**: reconstruye `(father + 1.08·mother)/2` y mide su diferencia máxima frente a la columna original (sin `assert`: aquí solo se observa).

In [ ]:
# Estructura del dataset y verificacion de la formula de la mid-parent
print("Filas x columnas:", galton.shape)
print("Familias unicas :", galton["family"].nunique())
print(galton[["father", "mother", "midparentHeight", "gender", "childHeight"]].head())

midparent_calc = (galton["father"] + 1.08 * galton["mother"]) / 2
dif_max = (midparent_calc - galton["midparentHeight"]).abs().max()
print(f"\nDiferencia maxima |formula - columna| = {dif_max:.10f}")
print("-> confirma que midparentHeight ya lleva el 1.08 del lado materno.")

**📖 Cómo se lee.** La diferencia máxima es prácticamente 0: `midparentHeight` **ya** aplica el 1,08 al lado de los padres. La respuesta `childHeight`, en cambio, queda cruda — la clave del momento de enseñanza sobre el género.

### 4.2 Modelo primario: `childHeight ~ midparentHeight` (cruda)

Se ajusta el OLS canónico documentado en `HistData` sobre las 934 filas crudas (ambos sexos). Es la réplica que reproduce directamente el «dos tercios» de Galton.

**❓ Qué se quiere averiguar.** ¿Qué parte de la ventaja de unos padres altos reaparece en el hijo? Es la pregunta de Galton, y la pendiente β₁ es su respuesta numérica.

- **Qué decide:** si la transmisión fuera completa, los extremos se perpetuarían y la dispersión de la población aumentaría generación tras generación; si es parcial, toda peculiaridad familiar se reabsorbe y la población permanece estable.
- **Antes de mirar el resultado:** si **β₁ ≈ 1**, el hijo hereda toda la desviación. Si **β₁ ≈ 0**, la altura de los padres no dice nada. Si **0 < β₁ < 1**, hay herencia pero atenuada — y ese «cuánto» es lo que se está midiendo.

**🔎 Qué hace este código.** Ajusta el OLS primario `childHeight ~ midparentHeight` con `statsmodels` y muestra el `summary()` completo (coeficientes, errores estándar, t, p, IC y diagnósticos del pie).

In [ ]:
# Replica PRIMARIA: OLS childHeight (cruda) ~ midparentHeight
x_g = galton["midparentHeight"].to_numpy()
y_g = galton["childHeight"].to_numpy()

modelo_galton = sm.OLS(y_g, sm.add_constant(x_g)).fit()
print(modelo_galton.summary())

**🔎 Qué hace este código.** Extrae del modelo los coeficientes y el ajuste: intercepto β₀, pendiente β₁, correlación r y R².

In [ ]:
# Coeficientes y ajuste de la replica primaria
b0_g, b1_g = modelo_galton.params
r_g = float(np.corrcoef(x_g, y_g)[0, 1])
r2_g = float(modelo_galton.rsquared)

print(f"Intercepto b0 = {b0_g:.4f} pulgadas")
print(f"Pendiente  b1 = {b1_g:.4f}")
print(f"r (Pearson)   = {r_g:.4f}")
print(f"R2            = {r2_g:.4f}")

**📖 Cómo se lee los coeficientes.**

- **β₀ = 22,64 pulgadas** es solo el **ancla** de la recta (la altura esperada si la mid-parent valiera 0, un escenario imposible): no se sobreinterpreta.
- **β₁ = 0,637** significa que por **cada pulgada adicional** de mid-parent, la altura esperada del hijo sube **0,64 pulgadas**. Como **0,64 < 1**, los hijos «regresan» hacia la media: heredan solo unos dos tercios de la ventaja o desventaja de sus padres. Esto es la **regresión a la media**.
- **R² = 0,103:** la mid-parent explica solo el ~10 % de la variación de la altura del hijo. Un R² bajo **no** invalida el modelo: confirma un hecho real (la herencia de estatura es débil a nivel individual) y es coherente con la regresión a la media.

### 🖐️ Cálculo manual — β₁, β₀ y R² con las fórmulas, sobre los datos de Galton

Se reconstruyen las tres cantidades del bloque 🧮 directamente desde `x_g`, `y_g` (sin llamar a `statsmodels`) y se verifica con `assert` que reproducen el modelo y los valores del contrato (β₁ = 0,6374, β₀ = 22,6362, R² = 0,103).

**🔎 Qué hace este código.** Calcula `Sₓᵧ = Σ(x−x̄)(y−ȳ)` y `Sₓₓ = Σ(x−x̄)²`, obtiene β₁ = Sₓᵧ/Sₓₓ y β₀ = ȳ − β₁x̄, y luego R² = 1 − SSE/SST. Comprueba con `assert` que igualan a `statsmodels` y a los targets del paper.

In [ ]:
# OLS a mano: b1, b0 y R2 con las formulas, verificados contra statsmodels
x_bar, y_bar = x_g.mean(), y_g.mean()
Sxy = np.sum((x_g - x_bar) * (y_g - y_bar))
Sxx = np.sum((x_g - x_bar) ** 2)
b1_mano = Sxy / Sxx                        # b1 = sum((x-xbar)(y-ybar)) / sum((x-xbar)^2)
b0_mano = y_bar - b1_mano * x_bar          # b0 = ybar - b1*xbar

y_hat_mano = b0_mano + b1_mano * x_g
SSE = np.sum((y_g - y_hat_mano) ** 2)
SST = np.sum((y_g - y_bar) ** 2)
r2_mano = 1 - SSE / SST                     # R2 = 1 - SSE/SST

print(f"b1 a mano = {b1_mano:.4f}   (statsmodels {b1_g:.4f})")
print(f"b0 a mano = {b0_mano:.4f}  (statsmodels {b0_g:.4f})")
print(f"R2 a mano = {r2_mano:.4f}   (statsmodels {r2_g:.4f})")

assert abs(b1_mano - b1_g) < 1e-8 and abs(b0_mano - b0_g) < 1e-8
assert abs(r2_mano - r2_g) < 1e-10
assert abs(b1_mano - 0.6374) < 1e-3 and abs(b0_mano - 22.6362) < 1e-3 and abs(r2_mano - 0.103) < 1e-3
print("\nOK: las formulas a mano reproducen b1=0.6374, b0=22.6362 y R2=0.103 (statsmodels).")

**📖 Cómo se lee.** Las tres cantidades reconstruidas manualmente igualan a `statsmodels` hasta el error de máquina: OLS **no es una caja negra**, es Σ(x−x̄)(y−ȳ)/Σ(x−x̄)². El `assert` deja constancia de que la mecánica del alumno reproduce el contrato.

### 📄 En el paper — la pendiente ≈ 2/3 de Galton

- **Galton, F. (1886).** *Regression Towards Mediocrity in Hereditary Stature*. Journal of the Anthropological Institute, **15**, 246-263. Galton reportó el *ratio of filial regression* = **2/3 ≈ 0,667** (a partir de datos tabulados y ajuste gráfico). El OLS moderno sobre filas individuales da **0,637**: la misma conclusión cualitativa (β₁ < 1, «regresión hacia la media»); la diferencia menor es de método (tabulación de 1886 vs. OLS de fila), no de error.
- **Hanley, J. A. (2004).** *"Transmuting" Women into Men: Galton's Family Data on Human Stature*. The American Statistician, **58**(3), 237-243. Recuperó los cuadernos originales de Galton y documentó la transmutación ×1,08 (base de la réplica secundaria de la Sección 4.4).

Procedencia completa en la ficha de la sesión de réplica del paper. El valor **operativo** prevalece (venv, 0,637); el **2/3 ≈ 0,667** es el *benchmark* histórico etiquetado.

**❓ Qué se quiere averiguar.** ¿Qué significa una pendiente menor que 1 traducida a un caso concreto: cuántas pulgadas mide el hijo de unos padres que están 4 pulgadas por encima de la media?

- **Por qué importa fuera de la estatura:** es uno de los errores de interpretación más costosos de la profesión. La sucursal que peor vendió el trimestre pasado tenderá a mejorar el siguiente **sin que se haga nada**, y quien haya aprobado una intervención en ese intervalo se atribuirá el mérito.
- **Antes de mirar el resultado:** si la predicción devolviera las 4 pulgadas completas, no habría regresión a la media y cualquier repunte sería atribuible a una causa; si devuelve **menos**, parte de todo repunte es puramente estadístico.

**🔎 Qué hace este código.** Ilustra la **regresión a la media** con un caso numérico: cuánto se predice para el hijo de una mid-parent situada 4 pulgadas por encima de la media.

In [ ]:
# Regresion a la media, ilustrada con un caso numerico
media_mid = x_g.mean()
media_child = y_g.mean()
desv = 4.0  # una mid-parent 4 pulgadas por encima de la media
pred_desv = b1_g * desv
print(f"Media de mid-parent : {media_mid:.2f} pulgadas")
print(f"Media de childHeight: {media_child:.2f} pulgadas")
print(f"\nUna mid-parent {desv:.0f} pulgadas SOBRE la media predice un hijo")
print(f"solo {pred_desv:.2f} pulgadas sobre la media de hijos ({b1_g:.3f} de la desviacion).")
print("-> los extremos se atenuan: regresion a la media.")

**📖 Cómo se lee.** Una ventaja de 4 pulgadas en los padres se traduce en solo ~2,55 pulgadas de ventaja esperada en el hijo (0,637 de la desviación): los extremos **se atenúan** hacia la media. **💡 En negocio:** un trimestre récord o una tienda estrella tienden a «empeorar» en la siguiente medición no por una intervención, sino por **reversión estadística** — atribuirlo a una campaña es el error de atribución causal que Galton identificó (el glosario de la sesión, «Sección 12»).

### 4.3 Momento de enseñanza: por qué Galton «transmutó»

Al dejar `childHeight` cruda, la correlación cae a r ≈ 0,32 porque la mezcla de sexos infla la varianza de la respuesta. La resolución pedagógica es aplicar la transmutación **también al hijo** (multiplicar por 1,08 la altura de las hijas), homogeneizando ambos sexos. Entonces r sube a ~0,50 (la cifra de Pearson) y la pendiente a ~0,71. **Ambos resultados son correctos**: el primario reproduce el «dos tercios» clásico, y el secundario explica por qué Galton transmutó.

**❓ Qué se quiere averiguar.** ¿Cuánto cambia la conclusión si se corrige el problema de diseño que Galton sí corrigió —mezclar en la misma variable estaturas de hijos y de hijas?

- **Qué está en juego:** distinguir un **hallazgo** de un **artefacto de la preparación de los datos**. Es el mismo riesgo que aparecerá al mezclar unidades, monedas o periodos en un dataset de negocio.
- **Antes de mirar el resultado:** si β₁ y r apenas se movieran, la mezcla de sexos sería irrelevante y la advertencia resultaría innecesaria. Si **se mueven de forma apreciable**, entonces una decisión de limpieza —invisible en el informe final— estaba cambiando el número que se reporta.

**🔎 Qué hace este código.** Aplica la transmutación ×1,08 a las hijas y reajusta el OLS sobre la misma mid-parent, comparando la pendiente y la correlación con las de la réplica primaria.

In [ ]:
# Replica SECUNDARIA: hijas x1.08 (transmutacion), OLS sobre la misma mid-parent
galton_t = galton.copy()
mask_f = galton_t["gender"].str.lower().str.startswith("f")
galton_t.loc[mask_f, "childHeight"] = galton_t.loc[mask_f, "childHeight"] * 1.08

y_gt = galton_t["childHeight"].to_numpy()
modelo_galton_t = sm.OLS(y_gt, sm.add_constant(x_g)).fit()
b0_gt, b1_gt = modelo_galton_t.params
r_gt = float(np.corrcoef(x_g, y_gt)[0, 1])

print(f"Transmutada  ->  b0 = {b0_gt:.4f}   b1 = {b1_gt:.4f}   r = {r_gt:.4f}   R2 = {modelo_galton_t.rsquared:.4f}")
print(f"Primaria     ->  b0 = {b0_g:.4f}   b1 = {b1_g:.4f}   r = {r_g:.4f}   R2 = {r2_g:.4f}")
print("\nHomogeneizar los sexos SUBE r (0.32 -> 0.50) y la pendiente (0.64 -> 0.71):")
print("por eso Galton 'transmutaba' las estaturas femeninas a equivalente masculino.")

**📖 Cómo se lee.** Con las hijas ×1,08 la pendiente sube a **0,7126** y r a ~0,50: exactamente lo que Hanley (2004) documenta y lo que Pearson halló. La pendiente **transmutada (0,7126)** y la **primaria (0,6374)** *enmarcan* el 2/3 histórico de Galton — ambas < 1, ambas «regresión a la media». El valor 0,7126 se guarda en el Excel como `pendiente_b1_transmutada`.

**🔎 Qué hace este código.** Traza la **dispersión cruda** `childHeight` vs. `midparentHeight` (figura EDA, directa de los datos, no del Excel).

In [ ]:
# Figura EDA: dispersion cruda childHeight vs midparentHeight
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_g, y_g, alpha=0.4, color=UPC_ROJO, edgecolor="none", s=25)
ax.set_xlabel("midparentHeight (pulgadas)")
ax.set_ylabel("childHeight (pulgadas)")
ax.set_title("Galton: dispersion cruda (datos originales)")
mostrar(fig, FIG_DIR / "galton_dispersion_cruda.png")

### 3.7 — ¿Cómo se verifica que el desempeño es real? Estabilidad fuera de muestra (train/test) (subsección 4.4)

Se cierra la réplica con el transversal: partición 80/20 (semilla 42) sobre el modelo primario. Si el error de prueba es estable y la pendiente de entrenamiento se parece a la global, el modelo es **fiable** (sin sobreajuste), precisamente lo esperable de un modelo simple.

**❓ Qué se quiere averiguar.** ¿El modelo describe solo las 934 familias que ya vio, o acierta también sobre familias que no usó para ajustarse?

- **Qué decide:** un modelo que solo describe sirve para explicar el pasado; uno que predice sirve para tomar decisiones sobre el futuro, que es el propósito habitual de un encargo analítico.
- **Antes de mirar el resultado:** si el error fuera de muestra fuese **mucho peor** que el de dentro, el modelo habría sobreajustado el ruido de la muestra. Si ambos quedan **parecidos**, el modelo es estable y su error es el costo real de usarlo.

**🔎 Qué hace este código.** Parte 80/20 **con semilla 42** (toda cifra de esta tabla es condicional a esa partición), ajusta `LinearRegression` en el train y evalúa en el test: pendiente de entrenamiento, RMSE, MAE y R² de prueba.

In [ ]:
# Train/test 80/20 (semilla 42) sobre la replica primaria
Xg = galton[["midparentHeight"]].to_numpy()
yg = galton["childHeight"].to_numpy()
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(Xg, yg, test_size=0.2, random_state=42)

lr_g = LinearRegression().fit(Xg_tr, yg_tr)
pend_train_g = float(lr_g.coef_[0])
yg_pred = lr_g.predict(Xg_te)
rmse_test_g = float(mean_squared_error(yg_te, yg_pred) ** 0.5)
mae_test_g = float(mean_absolute_error(yg_te, yg_pred))
r2_test_g = float(r2_score(yg_te, yg_pred))

print(f"Pendiente entrenamiento = {pend_train_g:.4f}  (global = {b1_g:.4f})")
print(f"RMSE test = {rmse_test_g:.3f} pulgadas")
print(f"MAE  test = {mae_test_g:.3f} pulgadas")
print(f"R2   test = {r2_test_g:.4f}")

**📖 Cómo se lee.** La pendiente de entrenamiento (0,644) casi iguala la global (0,637) y el error de prueba (RMSE 3,26, MAE 2,79 pulgadas) es del mismo orden que el de entrenamiento: modelo simple, **sin sobreajuste**.

**⚠️ Cifras condicionales a la semilla 42.** Cambiando la partición, la pendiente de entrenamiento se mueve entre **0,615 y 0,681** y el R² de prueba entre **0,045 y 0,132** (seis semillas; hoja `estabilidad_semillas` del Excel). Lo **robusto** es la conclusión —RMSE de entrenamiento ≈ RMSE de prueba, sin sobreajuste—, no el valor puntual: el R² de prueba **no** debe presentarse como confirmación del ~10 % in-sample.

## 3.8 — ¿Qué decisión habilita? Laboratorio de negocio: ventas ~ inversión en TV (*Advertising*) (Sección 5 del cuaderno)

**Dataset:** *Advertising* (ISLR, cap. 3): inversión publicitaria en TV, radio y prensa (en **miles de USD**) y **ventas** (en **miles de unidades**) en 200 mercados. Aquí se ajusta un **OLS simple** `sales ~ TV` (un solo predictor). El uso conjunto de radio y prensa (regresión múltiple) es materia de S04 y **no** se ajusta en esta sesión.

**🔎 Qué hace este código.** Prepara el dataset: descarta la columna índice sin nombre y resume `TV` y `sales`.

In [ ]:
# Preparacion de Advertising (descartar la columna indice sin nombre)
adv_cols = [c for c in adv.columns if not c.lower().startswith("unnamed")]
adv = adv[adv_cols]
print("Columnas de trabajo:", list(adv.columns))
print(adv[["TV", "sales"]].describe().round(2))

**❓ Qué se quiere averiguar.** ¿Cuántas unidades adicionales se venden por cada mil dólares invertidos en televisión, y con cuánta incertidumbre?

- **La decisión concreta:** con ese número se responde si conviene mover presupuesto hacia TV. Sin el intervalo, el número es una promesa sin garantía.
- **Antes de mirar el resultado:** si el intervalo de confianza de la pendiente **incluyera el 0**, no se podría afirmar siquiera que invertir en TV aumente las ventas. Si **excluye el 0**, queda por decidir si el efecto es lo bastante grande para pagar la inversión — significativo y rentable no son lo mismo.

**🔎 Qué hace este código.** Ajusta el OLS `sales ~ TV` con `statsmodels`, muestra el `summary()` y los coeficientes, y extrae **con precisión completa** la cadena de inferencia de β₁ (SE, t, p, F e IC 95 %), que el `summary()` solo imprime redondeada a tres decimales.

In [ ]:
# OLS simple: sales ~ TV
x_tv = adv["TV"].to_numpy()
y_sales = adv["sales"].to_numpy()

modelo_adv = sm.OLS(y_sales, sm.add_constant(x_tv)).fit()
b0_a, b1_a = modelo_adv.params
r2_a = float(modelo_adv.rsquared)
print(modelo_adv.summary())
print(f"\nb0 = {b0_a:.4f}   b1 = {b1_a:.4f}   R2 = {r2_a:.4f}")

# Inferencia de b1 con PRECISION COMPLETA (el summary redondea SE a 0.003 y el IC a [0.042, 0.053])
se_b1_a = float(modelo_adv.bse[1])
t_b1_a = float(modelo_adv.tvalues[1])
p_b1_a = float(modelo_adv.pvalues[1])
f_adv = float(modelo_adv.fvalue)
_ci_a = modelo_adv.conf_int()
ic95_b1_a = (float(_ci_a[1][0]), float(_ci_a[1][1]))
print(f"SE(b1) = {se_b1_a:.7f}   t = {t_b1_a:.3f}   p = {p_b1_a:.3e}   F = {f_adv:.2f}")
print(f"IC 95% de b1 = [{ic95_b1_a[0]:.6f} ; {ic95_b1_a[1]:.6f}]")

**📖 Interpretación en unidades de negocio.**

- **β₁ = 0,0475:** por cada **1000 USD adicionales** invertidos en TV, las ventas esperadas suben **~0,0475 miles de unidades ≈ 47,5 unidades**. Traducido a la gerencia: cada 1000 USD de TV se asocian con unas **47 unidades vendidas más**.
- **β₀ = 7,03:** ventas esperadas (~7030 unidades) con inversión **cero** en TV. Como la inversión mínima observada es ≈ 0,7 mil USD, `TV = 0` queda **al borde del rango observado**: se lee como venta base solo **con cautela** y **no** debe presentarse como una predicción firme fuera de los datos (extrapolación).
- **R² = 0,61:** la inversión en TV explica ~61 % de la variación de las ventas entre mercados; el 39 % restante depende de otros factores (radio, prensa, estacionalidad) que este modelo simple no incluye.

**🔎 Qué hace este código.** Parte 80/20 (**semilla 42**) y mide el error dentro y **fuera de muestra** del laboratorio: RMSE, MAE y R² de entrenamiento y de prueba, más el tamaño de cada bloque (en miles de unidades).

In [ ]:
# Train/test 80/20 (semilla 42): error dentro y fuera de muestra
Xa = adv[["TV"]].to_numpy()
ya = adv["sales"].to_numpy()
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.2, random_state=42)

lr_a = LinearRegression().fit(Xa_tr, ya_tr)
pend_train_a = float(lr_a.coef_[0])
ya_tr_pred, ya_te_pred = lr_a.predict(Xa_tr), lr_a.predict(Xa_te)
rmse_train_a = float(mean_squared_error(ya_tr, ya_tr_pred) ** 0.5)
rmse_test_a = float(mean_squared_error(ya_te, ya_te_pred) ** 0.5)
mae_train_a = float(mean_absolute_error(ya_tr, ya_tr_pred))
mae_test_a = float(mean_absolute_error(ya_te, ya_te_pred))
r2_train_a = float(r2_score(ya_tr, ya_tr_pred))
r2_test_a = float(r2_score(ya_te, ya_te_pred))
n_train_a, n_test_a = int(ya_tr.size), int(ya_te.size)

print(f"n train = {n_train_a}   n test = {n_test_a}   (semilla 42)")
print(f"Pendiente entrenamiento = {pend_train_a:.5f}  (global = {b1_a:.5f})")
print(f"RMSE train = {rmse_train_a:.3f}   RMSE test = {rmse_test_a:.3f}   miles de unidades")
print(f"MAE  train = {mae_train_a:.3f}   MAE  test = {mae_test_a:.3f}   miles de unidades")
print(f"R2   train = {r2_train_a:.4f}   R2   test = {r2_test_a:.4f}")

**📖 Cómo se lee.** RMSE train (3,26) y test (3,19) son cercanos: el modelo **generaliza sin sobreajuste evidente**. El MAE de prueba (2,44 miles de unidades) es la métrica más sencilla de comunicar a la gerencia («en promedio se yerra ~2 440 unidades»).

**⚠️ Todas estas cifras son de la semilla 42.** Con otras particiones el RMSE de prueba se mueve entre **2,94 y 3,30** y el R² de prueba entre **0,415 y 0,678** (hoja `estabilidad_semillas`). Lo que resiste en las seis semillas es «RMSE de entrenamiento ≈ RMSE de prueba»; **no** resiste la frase «el error de prueba es igual o mejor que el de entrenamiento», que falla en 2 de 6 particiones.

**❓ Qué se quiere averiguar.** Cuando un gerente pregunta «¿cuánto venderé si invierto 150 mil en TV?», ¿qué intervalo hay que darle: el de la media o el de una predicción individual?

- **Por qué es el error más costoso de la sesión:** entregar el intervalo de la media como si fuera el de una campaña concreta promete una precisión que el modelo no tiene, y el incumplimiento se detecta cuando ya no admite corrección.
- **Antes de mirar el resultado:** el intervalo de **predicción** debe resultar claramente **más ancho** que el de la media, porque carga además con la variabilidad individual. Si alguien los confunde, el ancho del compromiso cambia por completo.

**🔎 Qué hace este código.** Compara, en `TV = 150` mil USD, el **IC de la media** (incertidumbre de la recta) con el **intervalo de predicción** (suma la variabilidad irreducible del error). Repite el mismo contraste sobre la recta de **Galton** en la mid-parent promedio. Es una demostración didáctica; no escribe en el Excel.

In [ ]:
# IC de la media vs. INTERVALO DE PREDICCION en un punto (TV = 150 mil USD)
tv0 = 150.0
pred = modelo_adv.get_prediction([1, tv0])
sf = pred.summary_frame(alpha=0.05)
yhat0 = float(sf["mean"].iloc[0])
ic_med = (float(sf["mean_ci_lower"].iloc[0]), float(sf["mean_ci_upper"].iloc[0]))
ic_pred = (float(sf["obs_ci_lower"].iloc[0]), float(sf["obs_ci_upper"].iloc[0]))

print(f"Para TV = {tv0:.0f} mil USD, prediccion puntual y_hat = {yhat0:.2f} miles de unidades")
print(f"IC 95% de la MEDIA      : [{ic_med[0]:.2f} ; {ic_med[1]:.2f}]  (ancho {ic_med[1]-ic_med[0]:.2f})")
print(f"INTERVALO de PREDICCION : [{ic_pred[0]:.2f} ; {ic_pred[1]:.2f}]  (ancho {ic_pred[1]-ic_pred[0]:.2f})")

# Mismo contraste sobre la recta de GALTON, en la mid-parent promedio x0 = mean(midparentHeight)
x0_g = float(galton["midparentHeight"].mean())
sf_g = modelo_galton.get_prediction([1, x0_g]).summary_frame(alpha=0.05)
yhat0_g = float(sf_g["mean"].iloc[0])
ic_med_g = (float(sf_g["mean_ci_lower"].iloc[0]), float(sf_g["mean_ci_upper"].iloc[0]))
ic_pred_g = (float(sf_g["obs_ci_lower"].iloc[0]), float(sf_g["obs_ci_upper"].iloc[0]))
print(f"\n[Galton] mid-parent = {x0_g:.2f}: y_hat = {yhat0_g:.2f} pulgadas")
print(f"[Galton] IC 95% de la MEDIA      : [{ic_med_g[0]:.2f} ; {ic_med_g[1]:.2f}]  (ancho {ic_med_g[1]-ic_med_g[0]:.2f})")
print(f"[Galton] INTERVALO de PREDICCION : [{ic_pred_g[0]:.2f} ; {ic_pred_g[1]:.2f}]  (ancho {ic_pred_g[1]-ic_pred_g[0]:.2f})")

**📖 Cómo se lee.** El intervalo de predicción es **varias veces más ancho** que el IC de la media. **⚠️ Error costoso:** usar el IC de la media (estrecho) para comprometerse con **un** mercado concreto transmite falsa precisión. El IC de la media sirve para la planeación agregada (ventas medias de mercados con esa inversión); el de predicción, para un mercado individual.

**Recomendación gerencial (borrador).** La inversión en TV tiene una relación positiva, significativa y económicamente relevante con las ventas (~47 unidades por cada 1000 USD), y el modelo generaliza de forma estable. Es una guía razonable para presupuestar TV, pero **no** basta por sí sola: explica ~61 % de la variación y el diagnóstico (Sección 8) sugiere revisar los mercados de inversión alta. Mezclar canales (TV + radio + prensa) requiere regresión múltiple (S04). Y un β significativo **no prueba causalidad**: la relación es correlacional hasta calibrarla con experimentos.

## Transversal — Exportación a Excel y figuras de resultados (Sección 6 del cuaderno)

Convención del curso: **todo resultado del modelo se exporta a `resultados/S03_resultados.xlsx`** y las **figuras de resultados se generan LEYENDO ese Excel** (no desde objetos en memoria). Las figuras de EDA crudo (dispersión, residuales, Q-Q, cuarteto) se trazan directamente de los datos.

**🔎 Qué hace este código.** Vuelca al Excel de contrato las **11 hojas** de la sesión. Las **6 de contrato** —`regresion_galton`, `inferencia_galton`, `traintest_galton`, `advertising_ols`, `anscombe` y `advertising_canales`— conservan **verbatim** sus valores; se añaden **5 hojas de registro** que dan celda a las cifras que la guía, la clave y el deck citaban sin fuente: `metricas_train_test` (train/test completo de *Advertising*), `inferencia_advertising` (SE/t/p/IC clásicos, robustos HC0/HC3 y el abanico de residuales por tercios de TV), `rendimiento_canales` (efecto marginal, umbral de rentabilidad y ajuste de los tres canales), `estabilidad_semillas` (los titulares de train/test en seis particiones) y `orden_independencia` (Durbin-Watson original y permutado, orden del archivo e ICC intrafamiliar). **Es la única celda que escribe el Excel;** todas las figuras de resultados se generan leyéndolo.

In [ ]:
# Escritura del Excel de resultados (openpyxl), respetando el contrato de hojas
from openpyxl import Workbook
from statsmodels.stats.stattools import durbin_watson

# =====================================================================================
# REGISTRO AMPLIADO. Se computa aqui porque el Excel se escribe SOLO en esta celda.
# Da CELDA a las cifras que la guia, la clave de calificacion y el deck citan: metricas
# completas de train/test, inferencia de Advertising (clasica y robusta), rendimiento
# por canal, estabilidad entre semillas y orden/independencia de las filas.
# =====================================================================================

# (1) Advertising: inferencia robusta a heterocedasticidad (HC0/HC3).
#     REGISTRO, no contenido de S03: el tratamiento formal de errores robustos es S04.
_rob = {cv: sm.OLS(y_sales, sm.add_constant(x_tv)).fit(cov_type=cv) for cv in ("HC0", "HC3")}
se_hc0_a, t_hc0_a = float(_rob["HC0"].bse[1]), float(_rob["HC0"].tvalues[1])
se_hc3_a, t_hc3_a = float(_rob["HC3"].bse[1]), float(_rob["HC3"].tvalues[1])

# (2) Advertising: SD de los residuales por TERCIOS equifrecuentes de TV (el abanico, en cifras).
_res_adv = modelo_adv.resid
corte_tv_1, corte_tv_2 = float(np.quantile(x_tv, 1 / 3)), float(np.quantile(x_tv, 2 / 3))
_terciles = [_res_adv[x_tv <= corte_tv_1],
             _res_adv[(x_tv > corte_tv_1) & (x_tv <= corte_tv_2)],
             _res_adv[x_tv > corte_tv_2]]
sd_terciles = [float(np.std(g, ddof=1)) for g in _terciles]
n_terciles = [int(g.size) for g in _terciles]

# (3) Rendimiento por canal: efecto marginal, umbral de rentabilidad y ajuste.
#     sales esta en MILES de unidades y la inversion en MILES de USD:
#     b1 * 1000 = unidades por cada 1 000 USD;  1 000 / (b1 * 1000) = USD por unidad vendida.
rendimiento = []
for _canal in ["TV", "radio", "newspaper"]:
    _xc = adv[_canal].to_numpy(float)
    _mc = sm.OLS(y_sales, sm.add_constant(_xc)).fit()
    _b1c = float(_mc.params[1])
    rendimiento.append({"canal": _canal, "b1": _b1c,
                        "uds_por_1000usd": _b1c * 1000.0,
                        "umbral_usd_por_unidad": 1000.0 / (_b1c * 1000.0),
                        "r2": float(_mc.rsquared),
                        "r": float(np.corrcoef(_xc, y_sales)[0, 1]),
                        "t": float(_mc.tvalues[1])})
_rend = {d["canal"]: d for d in rendimiento}

# (4) Estabilidad de los titulares de train/test en 6 particiones 80/20.
SEMILLAS = [42, 0, 1, 7, 123, 2024]
estabilidad = []
for _nom, _X, _y in [("advertising", adv[["TV"]].to_numpy(), adv["sales"].to_numpy()),
                     ("galton", galton[["midparentHeight"]].to_numpy(), galton["childHeight"].to_numpy())]:
    for _s in SEMILLAS:
        _Xtr, _Xte, _ytr, _yte = train_test_split(_X, _y, test_size=0.2, random_state=_s)
        _m = LinearRegression().fit(_Xtr, _ytr)
        _ptr, _pte = _m.predict(_Xtr), _m.predict(_Xte)
        estabilidad.append({"dataset": _nom, "semilla": _s, "b1_train": float(_m.coef_[0]),
                            "rmse_train": float(mean_squared_error(_ytr, _ptr) ** 0.5),
                            "rmse_test": float(mean_squared_error(_yte, _pte) ** 0.5),
                            "mae_train": float(mean_absolute_error(_ytr, _ptr)),
                            "mae_test": float(mean_absolute_error(_yte, _pte)),
                            "r2_train": float(r2_score(_ytr, _ptr)),
                            "r2_test": float(r2_score(_yte, _pte))})
_est_adv = [e for e in estabilidad if e["dataset"] == "advertising"]
_est_gal = [e for e in estabilidad if e["dataset"] == "galton"]
n_test_peor = int(sum(1 for e in _est_adv if e["rmse_test"] > e["rmse_train"]))
brecha_max_rmse = max(abs(e["rmse_test"] - e["rmse_train"]) for e in _est_adv + _est_gal)

# (5) Orden e independencia de las filas: Galton (ordenado) vs. Advertising (independiente).
dw_galton = float(durbin_watson(modelo_galton.resid))
_rs = np.random.RandomState(42)   # RandomState: secuencia estable entre versiones de numpy
_dw_perm = [float(durbin_watson(modelo_galton.resid[_rs.permutation(modelo_galton.resid.size)]))
            for _ in range(200)]
dw_galton_perm1 = _dw_perm[0]
dw_galton_perm_media = float(np.mean(_dw_perm))
dw_galton_perm_p05 = float(np.percentile(_dw_perm, 5))
dw_galton_perm_p95 = float(np.percentile(_dw_perm, 95))
_idx_g = np.arange(len(galton))
corr_idx_father = float(np.corrcoef(_idx_g, galton["father"].to_numpy(float))[0, 1])
corr_idx_midparent = float(np.corrcoef(_idx_g, x_g)[0, 1])
_medias_fam = pd.DataFrame({"fam": galton["family"].to_numpy(),
                            "res": modelo_galton.resid}).groupby("fam")["res"].mean()
icc_familia = float(_medias_fam.var(ddof=0) / np.var(modelo_galton.resid))
n_familias = int(_medias_fam.size)
dw_adv = float(durbin_watson(_res_adv))
_idx_a = np.arange(len(adv))
corr_idx_tv = float(np.corrcoef(_idx_a, x_tv)[0, 1])
corr_idx_sales = float(np.corrcoef(_idx_a, y_sales)[0, 1])


def _num(v, d=3):
    "Formatea un numero con coma decimal (los literales de lectura se guardan en el Excel)."
    return f"{v:.{d}f}".replace(".", ",")


LECTURA_ORDEN_GALTON = (
    "En Galton el Durbin-Watson NO diagnostica autocorrelación temporal: el CSV está ordenado por "
    f"estatura paterna descendente (corr(índice, father) = {_num(corr_idx_father)}), y al permutar "
    f"las filas el DW pasa de {_num(dw_galton)} a {_num(dw_galton_perm_media)} (media de 200 "
    "permutaciones). La independencia entre observaciones está comprometida por familia "
    f"(hermanos: ICC intrafamiliar {_num(icc_familia, 2)}), no por secuencia."
)
LECTURA_ORDEN_ADV = (
    f"Advertising sí es independiente: índice consecutivo 1-200, corr(índice, TV) = "
    f"{_num(corr_idx_tv)} y DW = {_num(dw_adv)} (≈ 2)."
)
LECTURA_CANAL_AJUSTE = (
    f"TV es el canal de mayor AJUSTE simple: R² {_num(_rend['TV']['r2'])} frente a "
    f"{_num(_rend['radio']['r2'])} de radio y {_num(_rend['newspaper']['r2'])} de prensa."
)
LECTURA_CANAL_MARGINAL = (
    "radio es el canal de mayor RENDIMIENTO por dólar invertido: "
    f"{_num(_rend['radio']['uds_por_1000usd'], 1)} unidades por cada 1 000 USD frente a "
    f"{_num(_rend['TV']['uds_por_1000usd'], 1)} de TV, y su umbral de rentabilidad es "
    f"{_num(_rend['radio']['umbral_usd_por_unidad'], 2)} USD/unidad frente a "
    f"{_num(_rend['TV']['umbral_usd_por_unidad'], 2)} de TV: por efecto marginal TV es el PEOR "
    "de los tres."
)
AFIRMACION_RESISTE = (
    "RMSE de entrenamiento ≈ RMSE de prueba (sin sobreajuste): la brecha |RMSE test − RMSE train| "
    f"no supera {_num(brecha_max_rmse, 2)} en ninguna de las {len(SEMILLAS)} semillas."
)
AFIRMACION_NO_RESISTE = (
    "«El error de prueba es igual o mejor que el de entrenamiento» NO resiste: falla en "
    f"{n_test_peor} de {len(SEMILLAS)} semillas. El R² de prueba de Advertising varía "
    f"{_num(min(e['r2_test'] for e in _est_adv))}-{_num(max(e['r2_test'] for e in _est_adv))} "
    "según la partición: los valores puntuales no son publicables sin la etiqueta «semilla 42»."
)

wb = Workbook()

# --- Hoja 1: regresion_galton (CONTRATO EXACTO de celdas B2:B6; filas 7-9 = registro) ---
ws = wb.active
ws.title = "regresion_galton"
ws["A1"], ws["B1"] = "metrica", "valor"
ws["A2"], ws["B2"] = "pendiente_b1", round(float(b1_g), 6)
ws["A3"], ws["B3"] = "intercepto_b0", round(float(b0_g), 6)
ws["A4"], ws["B4"] = "r_pearson", round(float(r_g), 6)
ws["A5"], ws["B5"] = "r2", round(float(r2_g), 6)
ws["A6"], ws["B6"] = "pendiente_b1_transmutada", round(float(b1_gt), 6)
# Replica secundaria completa: la r y el R2 transmutados que el material publica ("r sube a 0,50").
ws["A7"], ws["B7"] = "r_pearson_transmutada", round(float(r_gt), 6)
ws["A8"], ws["B8"] = "r2_transmutada", round(float(modelo_galton_t.rsquared), 6)
ws["A9"], ws["B9"] = "intercepto_b0_transmutada", round(float(b0_gt), 6)

# --- Hoja 2: inferencia_galton ---
ci = modelo_galton.conf_int()  # ndarray: fila0 const, fila1 pendiente
ws2 = wb.create_sheet("inferencia_galton")
ws2["A1"], ws2["B1"] = "metrica", "valor"
infe = [
    ("se_b1", float(modelo_galton.bse[1])),
    ("t_b1", float(modelo_galton.tvalues[1])),
    ("p_b1", float(modelo_galton.pvalues[1])),
    ("f_statistic", float(modelo_galton.fvalue)),
    ("ic95_b1_inf", float(ci[1][0])),
    ("ic95_b1_sup", float(ci[1][1])),
    ("ic95_b0_inf", float(ci[0][0])),
    ("ic95_b0_sup", float(ci[0][1])),
]
for i, (k, v) in enumerate(infe, start=2):
    # El p-valor (~8e-24) se guarda con precision plena; el resto redondeado.
    ws2[f"A{i}"], ws2[f"B{i}"] = k, (v if k == "p_b1" else round(v, 6))

# --- Hoja 3: traintest_galton ---
ws3 = wb.create_sheet("traintest_galton")
ws3["A1"], ws3["B1"] = "metrica", "valor"
tt = [
    ("pendiente_train", pend_train_g),
    ("rmse_test", rmse_test_g),
    ("mae_test", mae_test_g),
    ("r2_test", r2_test_g),
]
for i, (k, v) in enumerate(tt, start=2):
    ws3[f"A{i}"], ws3[f"B{i}"] = k, round(float(v), 6)

# --- Hoja 4: advertising_ols ---
ws4 = wb.create_sheet("advertising_ols")
ws4["A1"], ws4["B1"] = "metrica", "valor"
adv_res = [
    ("b0", b0_a),
    ("b1", b1_a),
    ("r2", r2_a),
    ("rmse_train", rmse_train_a),
    ("rmse_test", rmse_test_a),
    ("mae_test", mae_test_a),
]
for i, (k, v) in enumerate(adv_res, start=2):
    ws4[f"A{i}"], ws4[f"B{i}"] = k, round(float(v), 6)

# --- Hoja 5: anscombe (tabla I-IV; columnas F y G = intercepto y R2, antes sin celda) ---
ws5 = wb.create_sheet("anscombe")
ws5.append(["dataset", "media_x", "media_y", "pendiente", "r", "intercepto_b0", "r2"])
for _, row in anscombe_stats.iterrows():
    ws5.append([row["dataset"], round(float(row["media_x"]), 6),
                round(float(row["media_y"]), 6), round(float(row["pendiente"]), 6),
                round(float(row["r"]), 6), round(float(row["intercepto_b0"]), 6),
                round(float(row["r2"]), 6)])

# --- Hoja 6: advertising_canales (R2 de cada canal por OLS SIMPLE separado; NO es regresion multiple) ---
# Alimenta la eleccion de canal del entregable: radio y prensa frente a TV, cada uno con UN predictor.
ws6 = wb.create_sheet("advertising_canales")
ws6.append(["canal", "b1", "r2", "r"])
for _d in rendimiento:
    ws6.append([_d["canal"], round(_d["b1"], 6), round(_d["r2"], 6), round(_d["r"], 6)])

# --- Hoja 7: metricas_train_test (REGISTRO) — train/test completo de Advertising, semilla 42 ---
ws7 = wb.create_sheet("metricas_train_test")
ws7["A1"], ws7["B1"] = "metrica", "valor"
for i, (k, v) in enumerate([
    ("adv_pendiente_train", pend_train_a), ("adv_rmse_train", rmse_train_a),
    ("adv_rmse_test", rmse_test_a), ("adv_mae_train", mae_train_a),
    ("adv_mae_test", mae_test_a), ("adv_r2_train", r2_train_a),
    ("adv_r2_test", r2_test_a), ("adv_n_train", n_train_a),
    ("adv_n_test", n_test_a), ("semilla", 42),
], start=2):
    ws7[f"A{i}"], ws7[f"B{i}"] = k, (v if isinstance(v, int) else round(float(v), 6))

# --- Hoja 8: inferencia_advertising (REGISTRO) — clasica, robusta HC0/HC3 y abanico por tercios ---
ws8 = wb.create_sheet("inferencia_advertising")
ws8["A1"], ws8["B1"] = "metrica", "valor"
for i, (k, v) in enumerate([
    ("se_b1", se_b1_a), ("t_b1", t_b1_a), ("p_b1", p_b1_a), ("f_statistic", f_adv),
    ("ic95_b1_inf", ic95_b1_a[0]), ("ic95_b1_sup", ic95_b1_a[1]),
    ("se_b1_hc0", se_hc0_a), ("t_b1_hc0", t_hc0_a),
    ("se_b1_hc3", se_hc3_a), ("t_b1_hc3", t_hc3_a),
    ("sd_resid_tercil_bajo", sd_terciles[0]), ("sd_resid_tercil_medio", sd_terciles[1]),
    ("sd_resid_tercil_alto", sd_terciles[2]),
    ("corte_tv_tercil_1", corte_tv_1), ("corte_tv_tercil_2", corte_tv_2),
    ("n_tercil_bajo", n_terciles[0]), ("n_tercil_medio", n_terciles[1]),
    ("n_tercil_alto", n_terciles[2]),
], start=2):
    ws8[f"A{i}"], ws8[f"B{i}"] = k, (v if isinstance(v, int) else
                                     (v if k == "p_b1" else round(float(v), 10)))
ws8["A20"], ws8["B20"] = "lectura_robustez", (
    f"El t clásico de sales ~ TV es {_num(t_b1_a, 2)}; recalculado con errores robustos de White "
    f"(HC3) baja a {_num(t_hc3_a, 2)} (HC0: {_num(t_hc0_a, 2)}): la significancia sobrevive a la "
    "heterocedasticidad. El t de 10,345 pertenece a Galton, no a Advertising.")

# --- Hoja 9: rendimiento_canales (REGISTRO) — el ranking depende del metro ---
ws9 = wb.create_sheet("rendimiento_canales")
ws9.append(["canal", "b1_marginal", "uds_por_1000usd", "umbral_usd_por_unidad", "r2", "r", "t"])
for _d in rendimiento:
    ws9.append([_d["canal"], round(_d["b1"], 6), round(_d["uds_por_1000usd"], 4),
                round(_d["umbral_usd_por_unidad"], 4), round(_d["r2"], 6),
                round(_d["r"], 6), round(_d["t"], 4)])
ws9["A6"], ws9["B6"] = "lectura_ajuste", LECTURA_CANAL_AJUSTE
ws9["A7"], ws9["B7"] = "lectura_marginal", LECTURA_CANAL_MARGINAL

# --- Hoja 10: estabilidad_semillas (REGISTRO) — los titulares de train/test en 6 particiones ---
ws10 = wb.create_sheet("estabilidad_semillas")
ws10.append(["dataset", "semilla", "b1_train", "rmse_train", "rmse_test",
             "mae_train", "mae_test", "r2_train", "r2_test"])
for _e in estabilidad:
    ws10.append([_e["dataset"], _e["semilla"], round(_e["b1_train"], 6),
                 round(_e["rmse_train"], 6), round(_e["rmse_test"], 6),
                 round(_e["mae_train"], 6), round(_e["mae_test"], 6),
                 round(_e["r2_train"], 6), round(_e["r2_test"], 6)])
ws10["A15"], ws10["B15"] = "resumen", "valor"
for i, (k, v) in enumerate([
    ("adv_rmse_test_min", min(e["rmse_test"] for e in _est_adv)),
    ("adv_rmse_test_max", max(e["rmse_test"] for e in _est_adv)),
    ("adv_r2_test_min", min(e["r2_test"] for e in _est_adv)),
    ("adv_r2_test_max", max(e["r2_test"] for e in _est_adv)),
    ("galton_b1_train_min", min(e["b1_train"] for e in _est_gal)),
    ("galton_b1_train_max", max(e["b1_train"] for e in _est_gal)),
    ("galton_r2_test_min", min(e["r2_test"] for e in _est_gal)),
    ("galton_r2_test_max", max(e["r2_test"] for e in _est_gal)),
    ("n_semillas", len(SEMILLAS)),
    ("adv_n_semillas_test_peor_que_train", n_test_peor),
    ("brecha_max_rmse_test_train", brecha_max_rmse),
], start=16):
    ws10[f"A{i}"], ws10[f"B{i}"] = k, (v if isinstance(v, int) else round(float(v), 6))
ws10["A28"], ws10["B28"] = "afirmacion_resiste", AFIRMACION_RESISTE
ws10["A29"], ws10["B29"] = "afirmacion_no_resiste", AFIRMACION_NO_RESISTE

# --- Hoja 11: orden_independencia (REGISTRO) — que mide realmente el Durbin-Watson aqui ---
ws11 = wb.create_sheet("orden_independencia")
ws11["A1"], ws11["B1"] = "metrica", "valor"
for i, (k, v) in enumerate([
    ("galton_dw_original", dw_galton),
    ("galton_dw_permutado_1a", dw_galton_perm1),
    ("galton_dw_permutado_media_200", dw_galton_perm_media),
    ("galton_dw_permutado_p05", dw_galton_perm_p05),
    ("galton_dw_permutado_p95", dw_galton_perm_p95),
    ("galton_corr_indice_father", corr_idx_father),
    ("galton_corr_indice_midparent", corr_idx_midparent),
    ("galton_icc_intrafamiliar", icc_familia),
    ("galton_n_familias", n_familias),
    ("advertising_dw", dw_adv),
    ("advertising_corr_indice_tv", corr_idx_tv),
    ("advertising_corr_indice_sales", corr_idx_sales),
], start=2):
    ws11[f"A{i}"], ws11[f"B{i}"] = k, (v if isinstance(v, int) else round(float(v), 6))
ws11["A15"], ws11["B15"] = "lectura_galton", LECTURA_ORDEN_GALTON
ws11["A16"], ws11["B16"] = "lectura_advertising", LECTURA_ORDEN_ADV

wb.save(XLSX)
print("Excel de resultados guardado:", XLSX)
for s in wb.sheetnames:
    print("  hoja:", s)
print("\n[registro] t clasico =", round(t_b1_a, 3), "| t HC3 =", round(t_hc3_a, 3))
print("[registro] rendimiento por 1 000 USD: TV", round(_rend["TV"]["uds_por_1000usd"], 1),
      "| radio", round(_rend["radio"]["uds_por_1000usd"], 1),
      "| prensa", round(_rend["newspaper"]["uds_por_1000usd"], 1))
# AJUSTE por canal (hoja `advertising_canales`): sale impreso para que el R2 de radio y prensa
# no quede solo en el Excel; es la cifra que la "nota de honestidad" del entregable presupone.
print("[registro] ajuste por canal (R2 de un OLS simple por canal): TV",
      round(_rend["TV"]["r2"], 3), "| radio", round(_rend["radio"]["r2"], 3),
      "| prensa", round(_rend["newspaper"]["r2"], 3))
print("[lectura]  el ranking depende del METRO: por AJUSTE gana TV (R2 "
      f"{_rend['TV']['r2']:.3f} vs. {_rend['radio']['r2']:.3f} de radio), pero por "
      "RENDIMIENTO MARGINAL gana radio "
      f"({_rend['radio']['uds_por_1000usd']:.1f} vs. {_rend['TV']['uds_por_1000usd']:.1f} "
      "uds por 1 000 USD): describir mejor no es rendir mas.")
print("[registro] DW Galton original", round(dw_galton, 3), "-> permutado",
      round(dw_galton_perm_media, 3), "| DW Advertising", round(dw_adv, 3))

**📖 Cómo se lee.** El Excel queda con 11 hojas. La hoja `regresion_galton` fija el contrato de la réplica (`pendiente_b1 = 0,637361`, `r2 = 0,103009`, `pendiente_b1_transmutada = 0,712585`) y registra además la réplica secundaria completa (`r_pearson_transmutada = 0,497012`, `r2_transmutada = 0,247021`, `intercepto_b0_transmutada = 19,917513`); `inferencia_galton` guarda `se_b1`, `t_b1` y el IC 95 % de β₁ `[0,5165 ; 0,7583]`. Las cinco hojas de registro cierran la trazabilidad: ninguna cifra publicada del material queda sin celda. La hoja `advertising_canales` registra el **ajuste de los tres canales por separado** (R² TV 0,612, radio 0,332, prensa 0,052), y la salida de la celda anterior la imprime junto al rendimiento marginal para dejar la lectura a la vista: **por ajuste destaca TV; por rendimiento por dólar, radio** (202,5 unidades por cada 1 000 USD frente a 47,5) — describir mejor no es rendir más, y por eso todo superlativo de canal debe declarar su metro (`rendimiento_canales!B6/B7`). La subsección 6.1 verifica que estos valores son **producto de la ejecución**.

### ✅ Verificación desde la base (subsección 6.1)

Antes de que el deck y la evaluación confíen en el Excel, se comprueba que ese registro es **producto de ejecutar el código sobre la base**, no un valor tecleado. Se **recomputa** desde los datos crudos ya cargados —de forma independiente del modelo ya ajustado— β₁, β₀, R², SE(β₁) y el IC 95 %, y se cruza con las celdas del Excel mediante `assert`. Refleja, visible y explicada, la lógica de el material de referencia de la sesión.

**🔎 Qué hace este código.** Recalcula β₁/β₀/R²/SE(β₁)/IC desde `midparentHeight` y `childHeight`, lee las hojas `regresion_galton` e `inferencia_galton` del Excel y comprueba con `assert` que **recomputado ≈ paper ≈ Excel**. No escribe en el Excel.

In [ ]:
# Verificacion desde la base: recomputar y cruzar con el Excel (NO escribe en el Excel)
from openpyxl import load_workbook

xb, yb = galton["midparentHeight"].to_numpy(), galton["childHeight"].to_numpy()
n = xb.size
Sxx_v = np.sum((xb - xb.mean()) ** 2)
b1_v = np.sum((xb - xb.mean()) * (yb - yb.mean())) / Sxx_v
b0_v = yb.mean() - b1_v * xb.mean()
yhat_v = b0_v + b1_v * xb
SSE_v = np.sum((yb - yhat_v) ** 2)
r2_v = 1 - SSE_v / np.sum((yb - yb.mean()) ** 2)
s_v = np.sqrt(SSE_v / (n - 2))
se_b1_v = s_v / np.sqrt(Sxx_v)
tcrit = stats.t.ppf(0.975, n - 2)
ic_b1 = (b1_v - tcrit * se_b1_v, b1_v + tcrit * se_b1_v)

wb_v = load_workbook(XLSX, data_only=True)
reg = {wb_v["regresion_galton"][f"A{r}"].value: wb_v["regresion_galton"][f"B{r}"].value for r in range(2, 7)}
inf = {wb_v["inferencia_galton"][f"A{r}"].value: wb_v["inferencia_galton"][f"B{r}"].value for r in range(2, 10)}

tabla = pd.DataFrame({
    "recomputado (venv)": [round(b1_v, 4), round(b0_v, 4), round(r2_v, 4), round(se_b1_v, 4), round(ic_b1[0], 4), round(ic_b1[1], 4)],
    "paper / referencia": [0.6374, 22.636, 0.103, 0.0616, 0.5165, 0.7583],
    "Excel (contrato)":   [reg["pendiente_b1"], reg["intercepto_b0"], reg["r2"], inf["se_b1"], inf["ic95_b1_inf"], inf["ic95_b1_sup"]],
}, index=["b1", "b0", "R2", "SE(b1)", "IC95 b1 inf", "IC95 b1 sup"])
print(tabla.to_string())

assert abs(b1_v - reg["pendiente_b1"]) < 1e-4
assert abs(b0_v - reg["intercepto_b0"]) < 1e-4
assert abs(r2_v - reg["r2"]) < 1e-4
assert abs(se_b1_v - inf["se_b1"]) < 1e-4
assert abs(ic_b1[0] - inf["ic95_b1_inf"]) < 1e-3 and abs(ic_b1[1] - inf["ic95_b1_sup"]) < 1e-3
assert abs(b1_v - 0.6374) <= 0.03 and abs(r2_v - 0.103) <= 0.01   # dentro de tolerancia del paper
print("\nOK: recomputado desde la base ~ paper (tolerancia) y ~ Excel (contrato).")

**📖 Cómo se lee.** Las tres columnas coinciden: lo **recomputado** desde los datos reproduce el **paper/target** (β₁ 0,637; R² 0,103) y coincide con el **Excel** hasta el cuarto decimal, incluidos SE(β₁) e IC. Los `assert` fallarían si alguien editara el Excel manualmente o si la base cambiara: por eso el registro es *auditable*. La versión ejecutable con red vive en el material de referencia de la sesión.

### Figuras de resultados (leídas del Excel) (subsección 6.2)

**🔎 Qué hace este código.** Lee las hojas `regresion_galton` y `advertising_ols` del Excel y genera tres figuras de resultados: pendiente primaria vs. transmutada, recta ajustada de Galton y recta ajustada de Advertising — todas con coeficientes **leídos del Excel**, no de objetos en memoria.

In [ ]:
# Figuras de RESULTADOS: se generan LEYENDO el Excel recien escrito
reg = pd.read_excel(XLSX, sheet_name="regresion_galton").set_index("metrica")["valor"]
adv_x = pd.read_excel(XLSX, sheet_name="advertising_ols").set_index("metrica")["valor"]

b1_xl = float(reg.loc["pendiente_b1"])
b1t_xl = float(reg.loc["pendiente_b1_transmutada"])
b0_xl = float(reg.loc["intercepto_b0"])
b0a_xl = float(adv_x.loc["b0"])
b1a_xl = float(adv_x.loc["b1"])

# (1) Barras: pendiente primaria vs. transmutada (valores del Excel)
fig, ax = plt.subplots(figsize=(6.5, 5))
barras = ax.bar(["Primaria\n(cruda)", "Transmutada\n(hijas x1.08)"],
                [b1_xl, b1t_xl], color=[UPC_ROJO, "#7A1420"], width=0.55)
ax.axhline(2/3, color=UPC_TINTA, ls="--", lw=1.2, label="Galton 1886 (2/3)")
for b, v in zip(barras, [b1_xl, b1t_xl]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)
ax.set_ylabel("Pendiente beta_1")
ax.set_title("Pendiente de Galton: cruda vs. transmutada")
ax.legend()
mostrar(fig, FIG_DIR / "galton_pendiente_primaria_vs_transmutada.png")

# (2) Dispersion Galton + recta ajustada (b0/b1 del Excel)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_g, y_g, alpha=0.35, color=UPC_ROJO, edgecolor="none", s=22)
xs = np.linspace(x_g.min(), x_g.max(), 100)
ax.plot(xs, b0_xl + b1_xl * xs, color="#1A1A1A", lw=2, label=f"y_hat = {b0_xl:.2f} + {b1_xl:.3f}*x")
ax.set_xlabel("midparentHeight (pulgadas)")
ax.set_ylabel("childHeight (pulgadas)")
ax.set_title("Galton: recta OLS ajustada (coeficientes del Excel)")
ax.legend()
mostrar(fig, FIG_DIR / "galton_recta_ajustada.png")

# (3) Dispersion Advertising + recta ajustada (b0/b1 del Excel)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_tv, y_sales, alpha=0.5, color=UPC_ROJO, edgecolor="none", s=28)
xs = np.linspace(x_tv.min(), x_tv.max(), 100)
ax.plot(xs, b0a_xl + b1a_xl * xs, color="#1A1A1A", lw=2, label=f"y_hat = {b0a_xl:.2f} + {b1a_xl:.4f}*TV")
ax.set_xlabel("Inversion en TV (miles de USD)")
ax.set_ylabel("Ventas (miles de unidades)")
ax.set_title("Advertising: recta OLS ajustada (coeficientes del Excel)")
ax.legend()
mostrar(fig, FIG_DIR / "advertising_recta_ajustada.png")

**📖 Cómo se lee.** Las tres figuras se reconstruyen desde el Excel: la línea discontinua marca el **2/3 de Galton**, que queda entre la pendiente cruda (0,637) y la transmutada (0,713). Cualquier cambio en el contrato del Excel se reflejaría automáticamente en estas figuras.

## 3.3 en profundidad — Construcción del OLS desde cero (Sección 7 del cuaderno)

El alumno **reconstruye el ajuste completo sin los objetos ya calculados**: del CSV crudo a β/R²/IC/predicción, resolviendo las **ecuaciones normales en forma matricial** `β = (XᵀX)⁻¹Xᵀy`. El objetivo es reproducir, de forma independiente, el **mismo contrato** del Excel. Si coincide (con `assert`), queda demostrado que **es el método —no un atajo— el que produce la réplica**.

**🔎 Qué hace este código.** Lee `GaltonFamilies.csv` desde cero, construye la matriz de diseño `X = [1, x]`, resuelve `β = (XᵀX)⁻¹Xᵀy` con álgebra lineal, y calcula R², SE(β₁) (de la matriz de varianzas-covarianzas `s²(XᵀX)⁻¹`), el IC 95 % y una predicción. Comprueba con `assert` que reproduce el Excel de contrato. No reutiliza `modelo_galton` ni reescribe el Excel.

In [ ]:
# Construye el OLS desde cero: del CSV crudo a beta/R2/IC/prediccion (NO escribe en el Excel)
from openpyxl import load_workbook

# 1) base cruda desde el CSV (sin reutilizar los modelos ya ajustados)
csv_galton = DATA_DIR / "GaltonFamilies.csv"
raw = pd.read_csv(csv_galton) if csv_galton.exists() else galton.copy()
xr = raw["midparentHeight"].to_numpy(float)
yr = raw["childHeight"].to_numpy(float)
n = xr.size

# 2) ecuaciones normales en forma matricial: beta = (XtX)^-1 Xt y
X = np.column_stack([np.ones(n), xr])
XtX = X.T @ X
beta = np.linalg.solve(XtX, X.T @ yr)      # [b0, b1]
b0_z, b1_z = beta

# 3) R2, error estandar de b1, IC 95% y una prediccion
yhat = X @ beta
SSE = np.sum((yr - yhat) ** 2)
r2_z = 1 - SSE / np.sum((yr - yr.mean()) ** 2)
sigma2 = SSE / (n - 2)
cov = sigma2 * np.linalg.inv(XtX)          # matriz de varianzas-covarianzas
se_b1_z = np.sqrt(cov[1, 1])
tcrit = stats.t.ppf(0.975, n - 2)
ic_b1_z = (b1_z - tcrit * se_b1_z, b1_z + tcrit * se_b1_z)
x0 = 70.0
pred_x0 = b0_z + b1_z * x0

print(f"b0 = {b0_z:.4f}   b1 = {b1_z:.4f}   R2 = {r2_z:.4f}")
print(f"SE(b1) = {se_b1_z:.4f}   IC95 b1 = [{ic_b1_z[0]:.4f} ; {ic_b1_z[1]:.4f}]")
print(f"prediccion para mid-parent = {x0:.0f}: y_hat = {pred_x0:.2f} pulgadas")

# 4) comprobar contra el Excel de contrato
c = load_workbook(XLSX, data_only=True)
reg_c = {c["regresion_galton"][f"A{r}"].value: c["regresion_galton"][f"B{r}"].value for r in range(2, 7)}
inf_c = {c["inferencia_galton"][f"A{r}"].value: c["inferencia_galton"][f"B{r}"].value for r in range(2, 10)}
assert abs(b1_z - reg_c["pendiente_b1"]) < 1e-4
assert abs(b0_z - reg_c["intercepto_b0"]) < 1e-4
assert abs(r2_z - reg_c["r2"]) < 1e-4
assert abs(se_b1_z - inf_c["se_b1"]) < 1e-4
assert abs(ic_b1_z[0] - inf_c["ic95_b1_inf"]) < 1e-3 and abs(ic_b1_z[1] - inf_c["ic95_b1_sup"]) < 1e-3
print("\nOK: el OLS desde cero reproduce el contrato (b1 0.6374 | b0 22.6362 | R2 0.103 | IC [0.5165; 0.7583]).")

**📖 Cómo se lee.** Partiendo del CSV crudo y sin reutilizar ningún objeto, el álgebra matricial reproduce el contrato exacto: β₁ = 0,6374, β₀ = 22,6362, R² = 0,103 e IC 95 % de β₁ = [0,5165 ; 0,7583]. La réplica no dependía de un estado oculto del cuaderno: el **método** la produce.

**✍️ Ahora, por cuenta propia.** Se propone repetir el ajuste (a) sobre la respuesta **transmutada** (hijas ×1,08) y comprobar que β₁ sube a ~0,7126; (b) calculando el **IC de β₀** y contrastándolo con `inferencia_galton` (`[14,27 ; 31,01]`); y (c) prediciendo para una mid-parent extrema (p. ej. 75 pulgadas) y discutiendo por qué **no** se debe extrapolar fuera del rango observado.

## 3.6 — ¿Cuándo se puede confiar en esta recta? Supuestos: cómo identificarlos y corregirlos (Sección 8 del cuaderno)

> **Fuente canónica:** la guía de supuestos de la sesión (qué es, cómo se identifica, cómo se corrige por método, alcance). Aquí se ejecutan los **diagnósticos**; su desarrollo teórico y sus fuentes viven en ese documento. **Ninguna celda de esta sección escribe en el Excel de contrato.**
>
> **Regla de alcance de S03.** El diagnóstico es sobre todo **visual** (gráficos de residuales, Q-Q, leverage) y de lectura del pie de `summary`. La **corrección** que S03 aplica de forma directa es una sola: **transformar** el predictor o la respuesta (log, potencias) dentro de un modelo que sigue siendo de **un** predictor. Las **pruebas formales** (RESET de Ramsey, Breusch-Pagan, Durbin-Watson, distancia de Cook) se **ejecutan y se leen con su umbral**, pero su **aplicación plena** (errores robustos, WLS, series, regresión robusta) es materia de **S04/S11** y aquí se **nombra**, no se corrige.

**Supuestos de Gauss-Markov (para que el OLS sea BLUE)** — `SUPUESTOS_S03.md`, Parte 1:

| Supuesto | Cómo identificar | Cómo corregir (método) | Alcance |
|---|---|---|---|
| **1.1 Linealidad** | Dispersión + residuales vs. ajustados (U/arco); RESET de Ramsey (`p<0,05`) | Transformar `X`/`Y` (log, √) — *polinomios/splines → S05* | Visual + transformar **S03** |
| **1.2 Exogeneidad `E[ε|X]=0`** | Razonar el dominio: variable omitida, causalidad inversa, regresión a la media (no hay test) | Incluir omitidas (múltiple, S04); aleatorización/causalidad (S12) | **S03**: advertir «asociación, no causa» |
| **1.3 Homocedasticidad** | Residuales vs. ajustados (abanico); escala-localización; Breusch-Pagan (`p<0,05`) | Transformar `Y` (log) — *ES robustos de White, WLS → S04* | Diagnóstico **S03** |
| **1.4 No autocorrelación** | Durbin-Watson (≈2 sano; `<1,5` positiva) **solo si hay orden**; residuales vs. orden | Newey-West/HAC; modelar la serie — *series → S11* | Detectar **S03** (solo con orden) |
| **1.5 Variación en X** | Simple: `X.var>0`. Múltiple: VIF → S04 | Simple: diseño con variación. Múltiple: Ridge (S05) | Parte simple **S03** |

**Supuesto de inferencia, diagnóstico de datos y metaprincipio** — `SUPUESTOS_S03.md`, Partes 2 y 3:

| Punto | Cómo identificar | Cómo corregir | Alcance |
|---|---|---|---|
| **2.1 Normalidad de los errores** | Q-Q plot (colas en S); Omnibus / Jarque-Bera (`p<0,05`) | Transformar `Y`; **TCL** en muestra grande — *bootstrap (avanzado)* | **S03** (leer Q-Q; TCL con n grande) |
| **2.2 Influyentes / leverage / atípicos** | Residuales vs. leverage; distancia de Cook (`>1` o `>4/n`); estudentizados (`|>3|`) | Investigar el dato; análisis de sensibilidad — *regresión robusta (avanzado)* | Diagnóstico + sensibilidad **S03** |
| **Parte 3 — Lección de Anscombe** | Graficar SIEMPRE (dispersión, residuales, Q-Q, leverage) antes de confiar en `R²`/`β`/`r` | Incorporar el diagnóstico gráfico como paso obligatorio | **S03 — central** |

**❓ Qué se quiere averiguar.** ¿Se puede confiar en los coeficientes y en los intervalos que se acaban de leer, o el modelo incumple los supuestos que los sostienen?

- **Por qué se realiza después y no antes:** los números se obtienen igual aunque el modelo sea inadecuado — el software no emite ninguna advertencia. El diagnóstico es lo único que separa un resultado válido de uno que solo lo parece.
- **Antes de mirar el resultado:** si los residuales aparecen **sin estructura**, repartidos alrededor de cero, la forma lineal es defendible. Si dibujan una **curva o un embudo**, la recta no es el modelo adecuado y los intervalos anteriores quedan en entredicho, con independencia del valor del R².

**🔎 Qué hace este código.** Diagnostica la **linealidad** (supuesto 1.1): traza los residuales vs. ajustados del modelo de Galton y ejecuta la **prueba RESET de Ramsey** (potencias del valor ajustado). Umbral: `p < 0,05` rechazaría la forma lineal.

In [ ]:
# Supuesto 1.1 (linealidad): residuales vs. ajustados + RESET de Ramsey (NO escribe en el Excel)
from statsmodels.stats.diagnostic import linear_reset

ajustados = modelo_galton.fittedvalues
residuales = modelo_galton.resid

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ajustados, residuales, alpha=0.4, color=UPC_ROJO, edgecolor="none", s=25)
ax.axhline(0, color=UPC_TINTA, lw=1.2, ls="--")
ax.set_xlabel("Valores ajustados (pulgadas)")
ax.set_ylabel("Residuales")
ax.set_title("Galton: residuales vs. ajustados")
mostrar(fig, FIG_DIR / "galton_residuales_vs_ajustados.png")

reset = linear_reset(modelo_galton, power=3, use_f=True)
print(f"RESET de Ramsey (potencias 2 y 3): F = {float(reset.fvalue):.3f}   p = {float(reset.pvalue):.4f}")
print("p >= 0.05 -> no se rechaza la forma lineal (sin curvatura no capturada).")

**📖 Cómo se lee.** La nube de residuales es **horizontal y sin forma** (linealidad y homocedasticidad razonables) y el RESET de Ramsey no es significativo: la recta es la forma adecuada para Galton. **Si** apareciera una U o arco, la corrección de S03 sería **transformar** el predictor (drill 3). Método en la guía de supuestos de la sesión («Parte 1.1»).

**🔎 Qué hace este código.** Diagnostica la **normalidad** (supuesto 2.1): traza el Q-Q plot de los residuales y calcula los estadísticos **Omnibus** y **Jarque-Bera** (los mismos del pie del `summary()`).

In [ ]:
# Supuesto 2.1 (normalidad): Q-Q plot + Omnibus / Jarque-Bera (NO escribe en el Excel)
from statsmodels.stats.stattools import jarque_bera, omni_normtest

fig, ax = plt.subplots(figsize=(7, 5))
stats.probplot(residuales, dist="norm", plot=ax)
ax.get_lines()[0].set_markerfacecolor(UPC_ROJO); ax.get_lines()[0].set_markeredgecolor("none")
ax.get_lines()[1].set_color(UPC_TINTA)
ax.set_title("Galton: Q-Q plot de los residuales")
mostrar(fig, FIG_DIR / "galton_qq.png")

jb_stat, jb_p, skew, kurt = jarque_bera(residuales)
omni_stat, omni_p = omni_normtest(residuales)
print(f"Omnibus = {float(omni_stat):.3f} (p = {float(omni_p):.2e})   Jarque-Bera = {float(jb_stat):.3f} (p = {float(jb_p):.2e})")
print("p < 0.05: desviacion leve de normalidad; con n = 934 el TCL protege la inferencia.")

**📖 Cómo se lee.** Los puntos caen casi sobre la diagonal, con colas levemente despegadas; Omnibus y Jarque-Bera son «significativos» pero eso es **esperable con n = 934**. **💡** La normalidad no hace falta para BLUE; solo para la inferencia exacta en muestra pequeña, y con `n` grande el **TCL** la relaja. Fuente: la guía de supuestos de la sesión («Parte 2.1»).

**🔎 Qué hace este código.** Diagnostica la **homocedasticidad** (supuesto 1.3): ejecuta la prueba de **Breusch-Pagan** en Galton (nube sana) y en Advertising (abanico) y grafica los residuales de Advertising, donde la varianza crece con las ventas ajustadas.

In [ ]:
# Supuesto 1.3 (homocedasticidad): Breusch-Pagan + residuales de Advertising (abanico) (NO escribe en el Excel)
from statsmodels.stats.diagnostic import het_breuschpagan

bp_g = het_breuschpagan(modelo_galton.resid, modelo_galton.model.exog)
bp_a = het_breuschpagan(modelo_adv.resid, modelo_adv.model.exog)
print(f"Breusch-Pagan Galton      : LM = {bp_g[0]:.3f}  p = {bp_g[1]:.4f}  (p>=0.05 -> homocedastico)")
print(f"Breusch-Pagan Advertising : LM = {bp_a[0]:.3f}  p = {bp_a[1]:.4f}  (p<0.05  -> heterocedastico)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(modelo_adv.fittedvalues, modelo_adv.resid, alpha=0.5, color=UPC_ROJO, edgecolor="none", s=30)
ax.axhline(0, color=UPC_TINTA, lw=1.2, ls="--")
ax.set_xlabel("Ventas ajustadas (miles de unidades)")
ax.set_ylabel("Residuales")
ax.set_title("Advertising: residuales vs. ajustados (sales ~ TV)")
mostrar(fig, FIG_DIR / "advertising_residuales_vs_ajustados.png")

**📖 Cómo se lee.** En **Galton** Breusch-Pagan no rechaza (nube sana); en **Advertising** sí: los residuales **abren un abanico** (la varianza crece con las ventas ajustadas). En cifras, la desviación estándar de los residuales por tercios de inversión en TV pasa de **1,86 → 2,98 → 4,42** (hoja `inferencia_advertising`): el error de un mercado grande casi triplica al de uno pequeño. **⚠️** La heterocedasticidad no sesga β₁, pero invalida errores estándar y p-valores: en `sales ~ TV` el t clásico **17,67** baja a **16,45** con errores robustos de White (HC3, registrados en la misma hoja) — la significancia sobrevive, pero el ajuste del error es real. En S03 la corrección posible es **transformar `Y`** (log); los **errores robustos** y WLS son la vía formal de **S04** (se nombran). Fuente: la guía de supuestos de la sesión («Parte 1.3»).

**🔎 Qué hace este código.** Diagnostica la **no autocorrelación** (supuesto 1.4). Calcula el **Durbin-Watson** de Galton tal como llega el archivo y **lo vuelve a calcular tras permutar las filas al azar** (200 permutaciones, semilla fija), mide cuánto está el archivo **ordenado por el predictor** (correlación entre el índice y `father`) y cuánto se parecen entre sí los residuales de **hermanos de una misma familia** (ICC intrafamiliar). Cierra con el contraste de *Advertising*, cuyas filas sí son independientes. La figura grafica los residuales frente al orden del archivo.

In [ ]:
# Supuesto 1.4 (no autocorrelacion): que mide REALMENTE el Durbin-Watson aqui (NO escribe en el Excel)
# Las cantidades vienen del registro calculado en la celda 60 (hoja `orden_independencia`).
print(f"Galton | DW tal como llega el archivo   = {dw_galton:.3f}")
print(f"Galton | DW tras permutar las filas     = {dw_galton_perm_media:.3f} "
      f"(media de 200 permutaciones; p05-p95 = {dw_galton_perm_p05:.3f}-{dw_galton_perm_p95:.3f})")
print(f"Galton | corr(indice del archivo, father)          = {corr_idx_father:.3f}")
print(f"Galton | corr(indice del archivo, midparentHeight) = {corr_idx_midparent:.3f}")
print(f"Galton | ICC intrafamiliar (varianza entre familias / varianza residual) = {icc_familia:.3f} "
      f"sobre {n_familias} familias")
print(f"\nAdvertising | DW = {dw_adv:.3f}   corr(indice, TV) = {corr_idx_tv:.3f}   "
      f"corr(indice, sales) = {corr_idx_sales:.3f}")
print("\nEl DW bajo de Galton es un ARTEFACTO del orden del archivo (estatura paterna descendente),")
print("no autocorrelacion temporal. Advertising, con indice sin relacion con las variables, da DW ~ 2.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
axes[0].plot(np.arange(modelo_galton.resid.size), modelo_galton.resid, lw=0.7, color=UPC_GRIS)
axes[0].axhline(0, color=UPC_TINTA, lw=1.0, ls="--")
axes[0].set_title(f"Orden del archivo (= estatura paterna descendente)\nDW = {dw_galton:.3f}",
                  fontsize=10)
axes[0].set_xlabel("Posicion en el archivo")
axes[0].set_ylabel("Residual")
_perm_fig = np.random.RandomState(42).permutation(modelo_galton.resid.size)
axes[1].plot(np.arange(modelo_galton.resid.size), modelo_galton.resid[_perm_fig], lw=0.7, color=UPC_GRIS)
axes[1].axhline(0, color=UPC_TINTA, lw=1.0, ls="--")
axes[1].set_title(f"Mismas observaciones, orden aleatorio\nDW = {dw_galton_perm1:.3f}", fontsize=10)
axes[1].set_xlabel("Posicion tras permutar")
fig.suptitle("Galton: el Durbin-Watson mide el ORDEN del archivo, no una secuencia temporal", fontsize=12)
mostrar(fig, FIG_DIR / "galton_residuales_vs_orden.png")

**📖 Cómo se lee.** **⚠️ Cautela clave, con la cifra delante:** Durbin-Watson solo se interpreta como autocorrelación cuando el orden de las filas **significa algo** (típicamente el tiempo). En Galton no lo significa, pero **tampoco es un orden neutro**: el archivo llega **ordenado por estatura paterna descendente** (corr(índice, `father`) = **−0,952**), y por eso los residuales vecinos se parecen. La prueba está en el panel derecho: **permutando las mismas 934 observaciones el DW sube de 1,386 a ≈ 1,99**. Es decir, el DW bajo es un **artefacto de la ordenación por el predictor**, no una dependencia serial que corregir.

**Lo que sí está comprometido aquí es la independencia, pero por otra vía: la familia.** Las 934 observaciones son hijos de solo **205 familias**; los residuales de hermanos se parecen entre sí (**ICC intrafamiliar ≈ 0,42**), de modo que las observaciones no son independientes **por agrupamiento**, no por secuencia. La consecuencia práctica es la misma que enseña la inferencia agrupada: los errores estándar «ingenuos» de β₁ resultan **optimistas** (el tamaño efectivo de muestra está más cerca de 205 familias que de 934 hijos). El tratamiento formal —errores agrupados, efectos aleatorios— excede S03; lo exigible aquí es **no leer el DW como autocorrelación** y **declarar el agrupamiento familiar**.

**Contraste:** los 200 mercados de *Advertising* sí son independientes —índice sin relación con las variables (corr(índice, TV) = 0,018) y **DW = 1,935 ≈ 2**—, y ahí el estadístico se lee con normalidad. En una serie de tiempo real, rachas largas del mismo signo delatarían dependencia serial; su tratamiento formal (Newey-West, modelos de series) es **S11**. Registro: hoja `orden_independencia` del Excel. Fuente: la guía de supuestos de la sesión («Parte 1.4»).

**🔎 Qué hace este código.** Diagnostica **observaciones influyentes** (supuesto 2.2): calcula la **distancia de Cook**, el **leverage** (`hᵢ`) y los **residuos estudentizados** del modelo de Galton, cuenta cuántos superan sus umbrales y grafica estudentizado vs. leverage.

In [ ]:
# Supuesto 2.2 (observaciones influyentes): distancia de Cook, leverage y estudentizados (NO escribe en el Excel)
infl = modelo_galton.get_influence()
cook = infl.cooks_distance[0]
leverage = infl.hat_matrix_diag
stud = infl.resid_studentized_internal
n = residuales.size
umbral_cook = 4.0 / n
umbral_lev = 2 * 2 / n     # 2p/n con p = 2 (intercepto + pendiente)

print(f"n = {n}   Cook maximo = {cook.max():.4f}  (umbral 4/n = {umbral_cook:.4f}; regla clasica Cook > 1)")
print(f"observaciones con Cook > 4/n      : {(cook > umbral_cook).sum()}")
print(f"observaciones con leverage > 2p/n : {(leverage > umbral_lev).sum()}")
print(f"residuos |estudentizado| > 3      : {(np.abs(stud) > 3).sum()}")
print("Cook maximo muy por debajo de 1: ningun punto secuestra la recta -> estimacion robusta.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(leverage, stud, alpha=0.4, color=UPC_ROJO, edgecolor="none", s=25)
ax.axhline(0, color=UPC_TINTA, lw=1.0, ls="--")
ax.axvline(umbral_lev, color=UPC_GRIS, lw=1.0, ls=":", label="leverage = 2p/n")
ax.set_xlabel("Leverage (h_i)")
ax.set_ylabel("Residuo estudentizado")
ax.set_title("Galton: residuo estudentizado vs. leverage")
ax.legend()
mostrar(fig, FIG_DIR / "galton_leverage.png")

**📖 Cómo se lee.** El **Cook máximo** queda muy por debajo de 1: aunque algunas observaciones superen el umbral sensible `4/n`, **ninguna domina** la recta. Un influyente combina residuo grande y alto leverage; el gráfico ubica esos casos en las esquinas. **💡** El protocolo de S03 es **investigar el dato** (¿error de captura?) y hacer **análisis de sensibilidad** (reajustar con y sin el caso); la regresión robusta es avanzada. Fuente: la guía de supuestos de la sesión («Parte 2.2»).

### 📄 En el paper — Anscombe (1973) y la lección de graficar

**Anscombe, F. J. (1973).** *Graphs in Statistical Analysis*. The American Statistician, **27**(1), 17-21. Cuatro conjuntos con `ŷ ≈ 3,00 + 0,50·X`, `r ≈ 0,816` y `R² ≈ 0,67` **idénticos**, pero con nubes opuestas: (I) lineal razonable, (II) curva (viola 1.1), (III) lineal con un atípico (2.2) y (IV) un único punto de leverage muy alto que crea toda la pendiente (2.2). Es el metaprincipio que articula la sesión: **graficar siempre** antes de confiar en `R²`/`β`/`r`.

**🔎 Qué hace este código.** Ajusta las **cuatro regresiones** de Anscombe (idénticas en pendiente y r) y **grafica las cuatro nubes** en una matriz 2×2, superponiendo cada recta. Es la demostración visual de por qué el resumen numérico no basta.

In [ ]:
# Leccion de Anscombe: 4 regresiones casi identicas, 4 nubes distintas -> SIEMPRE graficar
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, nombre in zip(axes.ravel(), ["I", "II", "III", "IV"]):
    g = anscombe[anscombe["dataset"] == nombre]
    x, y = g["x"].to_numpy(), g["y"].to_numpy()
    res = sm.OLS(y, sm.add_constant(x)).fit()
    xs = np.linspace(3, 20, 50)
    ax.scatter(x, y, color=UPC_ROJO, s=45, zorder=3, edgecolor="white")
    ax.plot(xs, res.params[0] + res.params[1] * xs, color=UPC_TINTA, lw=1.5)
    ax.set_title(f"Dataset {nombre}  (b1={res.params[1]:.2f}, r={np.corrcoef(x, y)[0, 1]:.3f})", fontsize=10)
    ax.set_xlim(3, 20); ax.set_ylim(2, 14); ax.set_xlabel("x"); ax.set_ylabel("y")
fig.suptitle("Cuarteto de Anscombe: mismos estadisticos, formas opuestas", fontsize=13)
mostrar(fig, FIG_DIR / "anscombe_cuarteto.png")

print(anscombe_stats.round(3).to_string(index=False))
print("\nMisma b1 (~0.50) y misma r (~0.816) en los 4... pero solo el I merece una recta.")

**📖 Cómo se lee.** Los cuatro tienen la **misma** pendiente (~0,50) y la **misma** r (~0,816), pero solo el dataset I admite una recta sin reservas; el II es curvo, el III tiene un atípico que desvía la recta y el IV un único punto de alto leverage que la genera. **La lección central de S03:** ningún `R²` ni `β` sustituye a mirar la nube. Fuente: la guía de supuestos de la sesión («Parte 3»).

## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Se resuelven en parejas; el detalle y la rúbrica están en `evaluacion/drills.docx`.

**Drill 1 — Interpretar β₁ en unidades de negocio.** A partir del modelo `sales ~ TV`, expresar la pendiente en la lengua de la gerencia (unidades vendidas por cada 1000 USD invertidos) y explicar, en una frase, qué decisión de presupuesto habilita. Añadir una **advertencia causal**: qué variable omitida (radio, prensa) podría estar contaminando β₁.

**Drill 2 — IC de la media vs. intervalo de predicción.** Sobre un mismo valor de inversión —**`TV = 150` mil USD**, el punto que resuelven `evaluacion/drills.docx` (anchos publicados 0,91 y 12,88)—, calcular ambos intervalos al 95 %, comparar sus anchos y decidir cuál corresponde a la pregunta «cuánto venderá *este* mercado en particular».

**Drill 3 — Diagnosticar un patrón curvo.** Ante un gráfico de residuales vs. ajustados con forma de U, identificar el supuesto violado (linealidad, 1.1) y proponer una solución **dentro del alcance de S03** (transformar el predictor o la respuesta), sin cambiar de familia de modelo.

## 3.9 — ¿Qué no se puede afirmar, y qué sigue en S04? Cierre (Sección 10 del cuaderno)

### Entregable evaluable
Ajustar e interpretar un **OLS simple** sobre *Advertising* (ventas ~ **un** canal a elección), reportando: β₀ y β₁ en unidades de negocio, R², diagnóstico de residuales y error fuera de muestra (train/test RMSE/MAE), con una **recomendación gerencial** para audiencia no técnica. Rúbrica vigesimal 0-20 en `evaluacion/entregable.docx`. Modalidad: parejas, entrega individual.

### Control corto
El **control corto** **no se resuelve en el aula**: es un quiz individual de ~18 minutos que se aplica como **tarea take-home**, se lanza en el cierre de la sesión (Min 89–90) y se entrega **antes de la clase siguiente**. Escala vigesimal: 10 preguntas × 2 puntos = 20. Cubre: interpretación de β₀/β₁, lectura de R² y del p-valor, diferencia IC de la media vs. intervalo de predicción, y lectura de residuales.

### Proyecto integrador
Este OLS alimenta la fase de **Modelado** del proyecto integrador: la partición train/test y las métricas RMSE/MAE introducidas aquí se reutilizan en S04-S14 para comparar modelos cada vez más complejos sobre el mismo problema de negocio.

### Materiales de apoyo de la sesión
- Guía del laboratorio: `laboratorio/GUIA_LABORATORIO_S03.docx`
- Plantilla de reporte gerencial: `plantillas/reporte_regresion_gerencial.docx`
- Guía de 1 página para leer residuales: `plantillas/guia_residuales.docx`
- Excel de regresión ya configurado: `plantillas/regresion_excel_configurada.xlsx`
- Drills y entregable: `evaluacion/drills.docx`, `evaluacion/entregable.docx`
- Mapa de celdas del cuaderno: el cuaderno de la sesión
- Supuestos de la sesión (fuente canónica): la guía de supuestos de la sesión (ver «Sección 8»)

### Para seguir explorando (actualidad — ver las fuentes de actualidad de la sesión)
- **Marketing Mix Modeling (MMM)** — Invoca (03/02/2025): el MMM usa regresión para relacionar inversión publicitaria y ventas; es la versión de negocio del laboratorio *Advertising*.
- **MMM en un mundo sin cookies** — Measured (24/06/2026): advierte sobre pasar de un enfoque correlacional a uno causal; refuerza que regresión ≠ causalidad.
- **Interpretar los coeficientes β de ventas** — OWOX (act. 17/06/2026): cómo leer β₁ en unidades de negocio (drill 1).
- **Reversión a la media en los mercados** — Nicola Wealth (13/02/2026): los desempeños extremos tienden a moderarse; la regresión a la media de Galton aplicada a inversiones.
- **El riesgo de extrapolar el desempeño excepcional** — The Investor's Podcast (act. 07/01/2026): sobrepagar por extrapolar tasas excepcionales que revierten a la media.

### Lecturas base
- **ISLR** cap. 3.1 y 3.3 (statlearning.com) — texto principal.
- **Galton, F. (1886)** — paper de la réplica.
- **Anscombe, F. J. (1973)** — *Graphs in Statistical Analysis*.
- **Hanley, J. A. (2004)** — reanálisis de los datos de Galton (transmutación 1,08).
- **Hastie, Tibshirani y Friedman — *The Elements of Statistical Learning* (ESL), cap. 3.2** — referencia avanzada del OLS (misma que el deck, slide 28).

Bibliografía completa verificada en la bibliografía de la sesión.

> **Alcance.** Esta sesión trabaja **regresión simple con un predictor** e introduce el transversal train/test + RMSE/MAE. La regresión múltiple, los errores robustos y k-fold son S04; la regularización y el modelado no lineal, S05; las series (autocorrelación como objeto) S11; la inferencia causal, S12. Aquí solo se nombran.